In [8]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo


KST = ZoneInfo("Asia/Seoul")


CASES = {

    "hanmi_20260612": {
        "company": "한미반도체",

        "event_start": datetime(
            2026, 6, 11,
            15, 30, 0,
            tzinfo=KST,
        ),

        "event_end": datetime(
            2026, 6, 12,
            15, 29, 59,
            tzinfo=KST,
        ),
    },


    "skhynix_20260304": {
        "company": "SK하이닉스",

        "event_start": datetime(
            2026, 3, 3,
            15, 30, 0,
            tzinfo=KST,
        ),

        "event_end": datetime(
            2026, 3, 4,
            15, 29, 59,
            tzinfo=KST,
        ),
    },
}

In [9]:
from pathlib import Path
import re

import pandas as pd


NEWS_STORE_DIR = Path(
    "./stored_data/news"
)


def get_case_paths(
    case_id: str,
):
    if case_id not in CASES:
        raise ValueError(
            f"등록되지 않은 case_id입니다: {case_id}"
        )

    case_dir = (
        NEWS_STORE_DIR
        / case_id
    )

    return {
        "case_dir":
            case_dir,

        "articles":
            case_dir
            / "articles.parquet",

        "embeddings":
            case_dir
            / "article_embeddings.npy",

        "embedding_metadata":
            case_dir
            / "embedding_metadata.parquet",

        "raw_html":
            case_dir
            / "raw_html",
    }

In [10]:
def normalize_body(
    body,
):
    """
    기사 본문의 의미는 변경하지 않고
    공백 문자만 정리합니다.

    - 연속 공백
    - 줄바꿈
    - 탭

    을 공백 하나로 통일합니다.
    """

    if not isinstance(
        body,
        str,
    ):
        return None


    normalized_body = re.sub(
        r"\s+",
        " ",
        body,
    ).strip()


    if not normalized_body:
        return None


    return normalized_body

In [11]:
def add_normalized_body(
    case_id: str,
):
    """
    해당 case의 articles.parquet에
    normalized_body 컬럼을 추가하여 저장합니다.

    기존 body 컬럼은 수정하지 않습니다.
    """

    paths = get_case_paths(
        case_id
    )

    articles_path = paths[
        "articles"
    ]


    if not articles_path.exists():

        raise FileNotFoundError(
            "articles.parquet을 찾을 수 없습니다: "
            f"{articles_path}"
        )


    articles_df = pd.read_parquet(
        articles_path
    )


    if "body" not in articles_df.columns:

        raise KeyError(
            "articles.parquet에 body 컬럼이 없습니다."
        )


    # =========================================
    # normalized_body 생성
    # =========================================

    articles_df[
        "normalized_body"
    ] = (
        articles_df[
            "body"
        ]
        .apply(
            normalize_body
        )
    )


    # =========================================
    # 저장
    # =========================================

    articles_df.to_parquet(
        articles_path,
        index=False,
    )


    # =========================================
    # 결과 확인
    # =========================================

    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "전체 기사 수:",
        len(
            articles_df
        )
    )

    print(
        "body 존재 기사 수:",
        articles_df[
            "body"
        ].notna().sum()
    )

    print(
        "normalized_body 존재 기사 수:",
        articles_df[
            "normalized_body"
        ].notna().sum()
    )

    print(
        "저장 경로:",
        articles_path
    )


    return articles_df

In [16]:
hanmi_articles = add_normalized_body(
    "hanmi_20260612"
)

case_id: hanmi_20260612
company: 한미반도체
전체 기사 수: 1079
body 존재 기사 수: 1077
normalized_body 존재 기사 수: 1077
저장 경로: stored_data/news/hanmi_20260612/articles.parquet


In [124]:
skhynix_articles = add_normalized_body(
    "skhynix_20260304"
)

case_id: skhynix_20260304
company: SK하이닉스
전체 기사 수: 18451
body 존재 기사 수: 18334
normalized_body 존재 기사 수: 18334
저장 경로: stored_data/news/skhynix_20260304/articles.parquet


In [17]:
hanmi_articles[
    [
        "article_id",
        "body",
        "normalized_body",
    ]
].head()

,article_id,body,normalized_body
0,A_08771cbb74756917,AI 국민배당금 소식에 장중 5% 급락\n마이크론 등 시간외거래 약세 영향도\n가파...,AI 국민배당금 소식에 장중 5% 급락 마이크론 등 시간외거래 약세 영향도 가파르게...
1,A_9e74e18abc5e5f55,"-외국인은 원전·바이오·조선 등 산업 재편 수혜주 집중\n-기관, 금융주 줄이고 반...","-외국인은 원전·바이오·조선 등 산업 재편 수혜주 집중 -기관, 금융주 줄이고 반도..."
2,A_cbe5778fc74f9c59,삼성전자와 SK하이닉스 등 반도체 대장주가 기염을 토하며 연일 사상 최고가 행진을 ...,삼성전자와 SK하이닉스 등 반도체 대장주가 기염을 토하며 연일 사상 최고가 행진을 ...
3,A_34d65ea6ee0752f9,‘8000피’ 눈앞서 상승 반납…반도체주 차익실현 매물 쏟아져\n국민배당금 논란·삼...,‘8000피’ 눈앞서 상승 반납…반도체주 차익실현 매물 쏟아져 국민배당금 논란·삼성...
4,A_e15a5c3cd38eae6c,"코스피, 연일 최고치 따른 차익실현 매물 급증\n트럼프, '해방 프로젝트' 거론 등...","코스피, 연일 최고치 따른 차익실현 매물 급증 트럼프, '해방 프로젝트' 거론 등 ..."


In [125]:
skhynix_articles[
    [
        "article_id",
        "body",
        "normalized_body",
    ]
].head()

,article_id,body,normalized_body
0,A_14163606272ff756,"AI 확산과 고성능 컴퓨팅, 첨단 패키징 수요 증가는 반도체 산업의 성장 공식을 바꾸고 있다. 경기 변동에 좌우되던 순환 산업을 넘어 구조적 성장 국면으로 전환되는 흐름 속에서, 세미콘 코리아 2026은 글로벌 반도체 생태계의 기술 진화 방향과 협력 전략을 한 자리에서 조망하는 산업의 기준점으로 주목된다.\n글로벌 반도체 생태계가 한자리에 모여 새로운 비즈니스 기회를 모색하고 기술 혁신의 방향성을 제시하는 ‘세미콘 코리아 2026(SEMICON KOREA 2026)’가 2월 11일부터 13일까지 3일간 서울 코엑스에서 열린다.\n올해 세미콘 코리아는 삼성전자와 SK하이닉스를 비롯해 소부장 전반이 동반 성장하고 있는 한국 반도체 산업의 위상을 반영하듯, 역대 최대 규모로 개최된다. 코엑스 A·B·C·D·E홀 전관은 물론 그랜드볼룸, 플라츠, 인근 웨스틴 서울 파르나스와 그랜드 인터컨티넨탈 서울 파르나스까지 행사 공간을 확장해 운영한다. 국내 반도체 산업의 빠른 성장으로 참가 수요가 전시장 수용 능력을 넘어선 데 따른 조치로, 산업 확장성을 상징적으로 보여주는 대목이다.\n이번 행사에는 약 550개 기업이 참여해 2,409개 부스를 운영하며, 3일간 약 7만 명 이상의 반도체 산업 관계자가 현장을 찾을 것으로 예상된다. 전시에는 △엔비디아 △삼성전자 △SK하이닉스 △인텔 △키오시아 △마이크론 △소니 등 글로벌 칩메이커와 팹리스 기업을 비롯해 △ASML △어플라이드 머티어리얼즈 △램리서치 △TEL △KLA 등 주요 장비·소재 기업이 대거 참가해 반도체 전 밸류체인을 아우르는 기술과 솔루션을 선보인다.\n이번 세미콘 코리아의 주제는 ‘TRANSFORM TOMORROW’로, AI가 촉발한 산업 패러다임 전환 속에서 미래 반도체 기술과 협력 모델을 조망하는 데 초점을 맞췄다. 행사 기간 동안 30여 개의 컨퍼런스가 열리며, 200여 명의 국내외 반도체 전문가가 연사로 참여해 △6대 공정 기반 첨단 제조 기술 △AI △스마트 매뉴팩처링 △계측·검사 △테스트 △화합물 전력 반도체 △사이버보안 △시장 트렌드 △투자 기회 △글로벌 R&D 협력 등 산업 전반의 핵심 이슈를 다룬다.\n세미콘 코리아 2026은 삼성전자, ASE, 케이던스 디자인 시스템즈, 램리서치, SK하이닉스, 엔비디아가 참여하는 기조연설로 막을 올린다. 기조연설에서는 AI 시스템 아키텍처, 첨단 패키징, 설계 자동화, 공정 혁신, 차세대 메모리, AI 슈퍼컴퓨팅 등 반도체 산업 전반의 핵심 이슈를 다루며, AI 시대를 관통하는 기술 진화 방향과 산업 전략을 제시할 예정이다.\n기술 컨퍼런스에서는 SEMI 테크놀로지 심포지움을 중심으로 소자·장비·소재·후공정 분야의 산업 및 학계 리더들이 모여 첨단 리소그래피와 재료, 식각, CMP·세정, 패키징 등 반도체 제조 핵심 기술의 현재와 미래를 조망한다. AI 서밋에서는 SEMI와 KAIST가 공동으로 AI 기반 산업 혁신 전략과 기술 로드맵을 공유하고, 테스트 포럼에서는 AI 시대에 대응하는 차세대 테스트 전략과 자동화·데이터 분석 기술을 집중 논의한다.\n이와 함께 화합물 전력 반도체 서밋에서는 와이드 밴드갭 반도체의 전 밸류체인과 시장 전망을, 사이버보안 포럼에서는 팹 보안과 데이터 거버넌스 등 반도체 산업의 핵심 보안 이슈를 다룬다. MI(계측·검사) 포럼과 스마트 매뉴팩처링 포럼에서는 제조 복잡성 심화에 대응한 계측·검사 기술과 AI·디지털 트윈 기반 자율 제조 혁신 사례가 공유되며, 글로벌 반도체 규제 전망 포럼에서는 국내외 규제와 에너지 정책 변화에 따른 컴플라이언스 전략을 짚는다.\n비즈니스 프로그램도 강화됐다. 마켓 트렌드 포럼을 통해 글로벌 반도체 시장과 기술 흐름을 종합 분석하고, 미국 반도체 투자 설명회에서는 미국 반도체 클러스터 및 지역 경제개발기구와의 교류를 통해 해외 투자 기회를 연결한다. 스타트업 서밋과 네덜란드-한국 반도체 기술 협력 세미나는 혁신 기술 교류와 글로벌 R&D 협력을 촉진하며, 구매상담회에서는 글로벌 반도체 기업과 국내 소부장 기업 간 약 100회의 비즈니스 미팅이 진행될 예정이다.\n미래 인재 육성을 위한 프로그램도 운영된다. ‘밋 더 익스퍼츠(Meet the Expert!)’ 멘토링을 통해 현직 엔지니어들이 이공계 학생들에게 직무와 커리어 조언을 제공하고, 기술 튜토리얼에서는 반도체 제조 핵심 공정을 중심으로 기초 기술 교육이 이뤄진다.\nSEMI 관계자는 “미세화 한계에 근접한 현시점에서 반도체 산업의 핵심 키워드는 ‘협력’”이라며 “공정과 기술 간 경계를 허물고 파트너 간 공동 로드맵을 구축하는 것이 경쟁력의 핵심이 되고 있다”고 밝혔다. 이어 “세미콘 코리아 2026을 계기로 글로벌 협력이 더욱 가속화되고, 새로운 파트너십과 혁신이 산업 전반으로 확산되기를 기대한다”고 말했다.","AI 확산과 고성능 컴퓨팅, 첨단 패키징 수요 증가는 반도체 산업의 성장 공식을 바꾸고 있다. 경기 변동에 좌우되던 순환 산업을 넘어 구조적 성장 국면으로 전환되는 흐름 속에서, 세미콘 코리아 2026은 글로벌 반도체 생태계의 기술 진화 방향과 협력 전략을 한 자리에서 조망하는 산업의 기준점으로 주목된다. 글로벌 반도체 생태계가 한자리에 모여 새로운 비즈니스 기회를 모색하고 기술 혁신의 방향성을 제시하는 ‘세미콘 코리아 2026(SEMICON KOREA 2026)’가 2월 11일부터 13일까지 3일간 서울 코엑스에서 열린다. 올해 세미콘 코리아는 삼성전자와 SK하이닉스를 비롯해 소부장 전반이 동반 성장하고 있는 한국 반도체 산업의 위상을 반영하듯, 역대 최대 규모로 개최된다. 코엑스 A·B·C·D·E홀 전관은 물론 그랜드볼룸, 플라츠, 인근 웨스틴 서울 파르나스와 그랜드 인터컨티넨탈 서울 파르나스까지 행사 공간을 확장해 운영한다. 국내 반도체 산업의 빠른 성장으로 참가 수요가 전시장 수용 능력을 넘어선 데 따른 조치로, 산업 확장성을 상징적으로 보여주는 대목이다. 이번 행사에는 약 550개 기업이 참여해 2,409개 부스를 운영하며, 3일간 약 7만 명 이상의 반도체 산업 관계자가 현장을 찾을 것으로 예상된다. 전시에는 △엔비디아 △삼성전자 △SK하이닉스 △인텔 △키오시아 △마이크론 △소니 등 글로벌 칩메이커와 팹리스 기업을 비롯해 △ASML △어플라이드 머티어리얼즈 △램리서치 △TEL △KLA 등 주요 장비·소재 기업이 대거 참가해 반도체 전 밸류체인을 아우르는 기술과 솔루션을 선보인다. 이번 세미콘 코리아의 주제는 ‘TRANSFORM TOMORROW’로, AI가 촉발한 산업 패러다임 전환 속에서 미래 반도체 기술과 협력 모델을 조망하는 데 초점을 맞췄다. 행사 기간 동안 30여 개의 컨퍼런스가 열리며, 200여 명의 국내외 반도체 전문가가 연사로 참여해 △6대 공정 기반 첨단 제조 기술 △AI △스마트 매뉴팩처링 △계측·검사 △테스트 △화합물 전력 반도체 △사이버보안 △시장 트렌드 △투자 기회 △글로벌 R&D 협력 등 산업 전반의 핵심 이슈를 다룬다. 세미콘 코리아 2026은 삼성전자, ASE, 케이던스 디자인 시스템즈, 램리서치, SK하이닉스, 엔비디아가 참여하는 기조연설로 막을 올린다. 기조연설에서는 AI 시스템 아키텍처, 첨단 패키징, 설계 자동화, 공정 혁신, 차세대 메모리, AI 슈퍼컴퓨팅 등 반도체 산업 전반의 핵심 이슈를 다루며, AI 시대를 관통하는 기술 진화 방향과 산업 전략을 제시할 예정이다. 기술 컨퍼런스에서는 SEMI 테크놀로지 심포지움을 중심으로 소자·장비·소재·후공정 분야의 산업 및 학계 리더들이 모여 첨단 리소그래피와 재료, 식각, CMP·세정, 패키징 등 반도체 제조 핵심 기술의 현재와 미래를 조망한다. AI 서밋에서는 SEMI와 KAIST가 공동으로 AI 기반 산업 혁신 전략과 기술 로드맵을 공유하고, 테스트 포럼에서는 AI 시대에 대응하는 차세대 테스트 전략과 자동화·데이터 분석 기술을 집중 논의한다. 이와 함께 화합물 전력 반도체 서밋에서는 와이드 밴드갭 반도체의 전 밸류체인과 시장 전망을, 사이버보안 포럼에서는 팹 보안과 데이터 거버넌스 등 반도체 산업의 핵심 보안 

In [14]:
import torch

from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = "BAAI/bge-m3"


if torch.cuda.is_available():
    EMBEDDING_DEVICE = "cuda"

elif (
    hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    EMBEDDING_DEVICE = "mps"

else:
    EMBEDDING_DEVICE = "cpu"


print(
    "Embedding model:",
    EMBEDDING_MODEL_NAME
)

print(
    "Embedding device:",
    EMBEDDING_DEVICE
)


embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=EMBEDDING_DEVICE,
)


tokenizer = embedding_model.tokenizer


print(
    "SentenceTransformer max_seq_length:",
    embedding_model.max_seq_length
)

Embedding model: BAAI/bge-m3
Embedding device: mps
SentenceTransformer max_seq_length: 8192


In [127]:
import numpy as np
import pandas as pd


def analyze_token_lengths(
    case_id: str,
    embedding_model,
    batch_size: int = 256,
):
    """
    해당 case의 normalized_body를
    현재 embedding model의 tokenizer 기준으로
    token 길이를 측정합니다.

    articles.parquet 자체는 수정하지 않습니다.
    """

    paths = get_case_paths(
        case_id
    )

    articles_path = paths[
        "articles"
    ]


    # =========================================
    # Articles 불러오기
    # =========================================

    articles_df = pd.read_parquet(
        articles_path
    )


    if "normalized_body" not in articles_df.columns:
        raise KeyError(
            "normalized_body 컬럼이 없습니다. "
            "먼저 add_normalized_body()를 실행해주세요."
        )


    # normalized_body가 있는 기사만 분석
    analysis_df = (
        articles_df[
            articles_df[
                "normalized_body"
            ].notna()
        ]
        .copy()
        .reset_index(drop=True)
    )


    tokenizer = (
        embedding_model.tokenizer
    )


    texts = (
        analysis_df[
            "normalized_body"
        ]
        .tolist()
    )


    token_lengths = []


    # =========================================
    # Batch 단위 Token 길이 계산
    # =========================================

    for start in range(
        0,
        len(texts),
        batch_size,
    ):

        batch_texts = texts[
            start:
            start + batch_size
        ]


        encoded = tokenizer(
            batch_texts,

            add_special_tokens=True,

            truncation=False,

            padding=False,

            return_length=True,
        )


        token_lengths.extend(
            encoded[
                "length"
            ]
        )


    analysis_df[
        "token_length"
    ] = token_lengths


    # =========================================
    # 요약 통계
    # =========================================

    lengths = (
        analysis_df[
            "token_length"
        ]
        .to_numpy()
    )


    summary_df = pd.DataFrame(
        {
            "metric": [
                "count",
                "mean",
                "median",
                "p75",
                "p90",
                "p95",
                "p99",
                "max",
            ],

            "token_length": [
                len(lengths),
                lengths.mean(),
                np.median(lengths),
                np.percentile(
                    lengths,
                    75,
                ),
                np.percentile(
                    lengths,
                    90,
                ),
                np.percentile(
                    lengths,
                    95,
                ),
                np.percentile(
                    lengths,
                    99,
                ),
                lengths.max(),
            ],
        }
    )


    max_seq_length = (
        embedding_model
        .max_seq_length
    )


    over_limit_count = (
        analysis_df[
            "token_length"
        ]
        > max_seq_length
    ).sum()


    over_limit_ratio = (
        over_limit_count
        / len(analysis_df)
        if len(analysis_df) > 0
        else 0
    )


    # =========================================
    # 출력
    # =========================================

    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "분석 기사 수:",
        len(
            analysis_df
        )
    )

    print(
        "BGE-M3 max_seq_length:",
        max_seq_length
    )

    print()

    print(
        summary_df.to_string(
            index=False
        )
    )

    print()

    print(
        "max_seq_length 초과 기사 수:",
        over_limit_count
    )

    print(
        "max_seq_length 초과 비율:",
        f"{over_limit_ratio:.2%}"
    )


    return (
        analysis_df,
        summary_df,
    )

In [23]:
hanmi_token_df, hanmi_token_summary = (
    analyze_token_lengths(
        case_id="hanmi_20260612",
        embedding_model=embedding_model,
    )
)

Token indices sequence length is longer than the specified maximum sequence length for this model (15054 > 8192). Running this sequence through the model will result in indexing errors


case_id: hanmi_20260612
company: 한미반도체
분석 기사 수: 1077
BGE-M3 max_seq_length: 8192

metric  token_length
 count   1077.000000
  mean    873.709378
median    735.000000
   p75   1036.000000
   p90   1395.600000
   p95   1666.600000
   p99   3190.360000
   max  15054.000000

max_seq_length 초과 기사 수: 3
max_seq_length 초과 비율: 0.28%


In [24]:
hanmi_token_df[
    [
        "article_id",
        "publisher_name",
        "title",
        "body_length",
        "token_length",
    ]
].sort_values(
    "token_length",
    ascending=False,
).head(20)

,article_id,publisher_name,title,body_length,token_length
38,A_d6747b6ba1bfb9d0,아시아타임즈,"[증권 뉴스브리핑] 대법 ""옵티머스펀드 판매 NH증권, JYP에 15억원 배상...",28774,15054
190,A_cbb589df0331625d,아시아타임즈,"[증권 뉴스브리핑] ""단일종목 레버리지, 하루 60% 손실 가능...'음의 복리...",22354,11707
618,A_a75b3b0c1546d7fa,포쓰저널,"[경제소식] 하나은행, 대한민국 축구국가대표팀 선전 기원 적금 출시",21404,11272
255,A_7adc61f0f0f68be4,포쓰저널,[경제소식] KB금융 '경기도어린이박물관' 문화 행사 개최,12137,6328
646,A_e3e8c84d68173324,톱스타뉴스,"월가 AI 돈줄, 한국 반도체·전력망까지 흔든다",9237,4819
474,A_5a6d3cd6518ab6ba,이코노미조선,"[재계 이슈] 미래에셋, 증권 업계 최초 '분기 순익 1조' 달성 | '한국판...",4848,4500
954,A_f005b3b3831d7924,핀포인트뉴스,"기업가치 제고 지표 가시화에 수급 폭발…코리아 밸류업 지수 테마, 사...",6823,3909
441,A_3b26cd71dfbbc108,토큰포스트,"[칼럼] 격동의 시대, 어떻게 투자할 것인가",7012,3854
613,A_1be149b943f430a3,위키리크스한국,[ETF NOW] 삼성자산운용·미래에셋운용·KB운용·한화운용·타임폴리오운...,7005,3820
890,A_736abfceace61de5,매일경제,美서 반도체 소부장 2배 ETF 나온다,6457,3460


In [128]:
skhynix_token_df, skhynix_token_summary = (
    analyze_token_lengths(
        case_id="skhynix_20260304",
        embedding_model=embedding_model,
    )
)

Token indices sequence length is longer than the specified maximum sequence length for this model (11399 > 8192). Running this sequence through the model will result in indexing errors


case_id: skhynix_20260304
company: SK하이닉스
분석 기사 수: 18334
BGE-M3 max_seq_length: 8192

metric  token_length
 count  18334.000000
  mean    859.195593
median    684.000000
   p75    974.750000
   p90   1369.000000
   p95   1793.000000
   p99   4295.050000
   max  19170.000000

max_seq_length 초과 기사 수: 57
max_seq_length 초과 비율: 0.31%


In [129]:
skhynix_token_df[
    [
        "article_id",
        "publisher_name",
        "title",
        "body_length",
        "token_length",
    ]
].sort_values(
    "token_length",
    ascending=False,
).head(20)

,article_id,publisher_name,title,body_length,token_length
7265,A_e7e1af9d8b7d2fcb,CATCH NEWS,2월 9일 주식시장 주요공시,33288,19170
1020,A_bf6d13dda66b3824,현대경제신문,"2차전지 관련주, '봄날의 햇살' 쎄노텍·쎄크·애경케미칼·엠플러스......",29701,16145
1031,A_2b0f634ca922d3b2,현대경제신문,"2차전지 관련주, '봄꽃만개' 롯데에너지머티리얼즈·씨아이에스·이녹스...",29418,15975
1075,A_1df82e786ebc4e38,현대경제신문,"로봇·산업용·협동로봇 관련주, '함박웃음' 계양전기·아이엘·씨피시...",25791,13978
1085,A_9bbad8856c142d4d,현대경제신문,"로봇·산업용·협동로봇 관련주, '웃음꽃' 계양전기·우림피티에스·씨...",24282,13173
1025,A_d8d9500e5857a1e9,현대경제신문,"로봇·산업용·협동로봇 관련주, '웃음꽃' TPC·대동기어·아진엑스텍·...",23875,13046
1064,A_0bff3dd8b3d841a7,현대경제신문,"로봇·산업용·협동로봇 관련주, '싱글벙글' 와이투솔루션·퍼스텍·아이...",23840,12985
1037,A_7b710fe90f7c93ee,현대경제신문,"로봇·산업용·협동로봇 관련주, '봄꽃만개' 대동기어·휴림로봇·대동...",23709,12960
6775,A_e80253277865c509,아시아타임즈,"[증권소식] 거래소, 첫 스타트업 인수/증권금융, 홍콩법인 출범식/무궁...",23224,12384
1086,A_915f62c57d6e1d3d,현대경제신문,"반도체 장비 관련주, '신바람난' 신성이엔지·HPSP·한미반도체·서진시...",23278,12295


In [25]:
bins = [
    0,
    512,
    1024,
    2048,
    4096,
    8192,
    float("inf"),
]

labels = [
    "0~512",
    "513~1024",
    "1025~2048",
    "2049~4096",
    "4097~8192",
    "8193+",
]


hanmi_token_df[
    "token_length_range"
] = pd.cut(
    hanmi_token_df[
        "token_length"
    ],

    bins=bins,

    labels=labels,

    include_lowest=True,
)


token_range_summary = (
    hanmi_token_df[
        "token_length_range"
    ]
    .value_counts(
        sort=False
    )
    .rename(
        "article_count"
    )
    .to_frame()
)


token_range_summary[
    "ratio"
] = (
    token_range_summary[
        "article_count"
    ]
    / len(
        hanmi_token_df
    )
)


token_range_summary

,article_count,ratio
token_length_range,,
0~512,276,0.256267
513~1024,527,0.489322
1025~2048,242,0.224698
2049~4096,26,0.024141
4097~8192,3,0.002786
8193+,3,0.002786


In [130]:
bins = [
    0,
    512,
    1024,
    2048,
    4096,
    8192,
    float("inf"),
]

labels = [
    "0~512",
    "513~1024",
    "1025~2048",
    "2049~4096",
    "4097~8192",
    "8193+",
]


skhynix_token_df[
    "token_length_range"
] = pd.cut(
    skhynix_token_df[
        "token_length"
    ],

    bins=bins,

    labels=labels,

    include_lowest=True,
)


token_range_summary = (
    skhynix_token_df[
        "token_length_range"
    ]
    .value_counts(
        sort=False
    )
    .rename(
        "article_count"
    )
    .to_frame()
)


token_range_summary[
    "ratio"
] = (
    token_range_summary[
        "article_count"
    ]
    / len(
        skhynix_token_df
    )
)


token_range_summary

,article_count,ratio
token_length_range,,
0~512,5184,0.282753
513~1024,9053,0.493782
1025~2048,3405,0.185721
2049~4096,499,0.027217
4097~8192,136,0.007418
8193+,57,0.003109


In [131]:
CHUNK_SIZE = 2048
CHUNK_OVERLAP = 256


def chunk_article_body(
    text: str,
    tokenizer,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
):
    """
    normalized_body를 tokenizer token 기준으로
    여러 chunk로 분할합니다.

    - chunk_size: 최대 2048 tokens
    - chunk_overlap: 인접 chunk 간 256 tokens 중첩
    - 기사 전체 내용을 끝까지 사용
    """

    if not isinstance(text, str):
        return []

    text = text.strip()

    if not text:
        return []


    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap은 chunk_size보다 작아야 합니다."
        )


    # special token을 제외한 실제 본문 token
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False,
    )


    if not token_ids:
        return []


    # 2048 tokens 이하라면 분할하지 않음
    if len(token_ids) <= chunk_size:

        return [
            tokenizer.decode(
                token_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            ).strip()
        ]


    chunks = []

    step_size = (
        chunk_size
        - chunk_overlap
    )


    start = 0


    while start < len(token_ids):

        end = min(
            start + chunk_size,
            len(token_ids),
        )


        chunk_token_ids = (
            token_ids[
                start:end
            ]
        )


        chunk_text = tokenizer.decode(
            chunk_token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()


        if chunk_text:
            chunks.append(
                chunk_text
            )


        # 마지막 token까지 처리했다면 종료
        if end >= len(token_ids):
            break


        start += step_size


    return chunks

In [132]:
def analyze_case_chunks(
    case_id: str,
    embedding_model,
):
    """
    해당 case의 normalized_body를
    공통 chunking 규칙으로 나누고
    기사별 chunk 개수를 계산합니다.
    """

    paths = get_case_paths(
        case_id
    )


    articles_df = pd.read_parquet(
        paths["articles"]
    )


    if "normalized_body" not in articles_df.columns:

        raise KeyError(
            "normalized_body 컬럼이 없습니다. "
            "먼저 add_normalized_body()를 실행해주세요."
        )


    tokenizer = (
        embedding_model.tokenizer
    )


    result_df = (
        articles_df
        .copy()
    )


    chunk_counts = []


    for text in result_df[
        "normalized_body"
    ]:

        chunks = chunk_article_body(
            text=text,
            tokenizer=tokenizer,
        )

        chunk_counts.append(
            len(chunks)
        )


    result_df[
        "chunk_count"
    ] = chunk_counts


    valid_chunk_counts = (
        result_df.loc[
            result_df[
                "chunk_count"
            ] > 0,
            "chunk_count",
        ]
    )


    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "전체 기사 수:",
        len(result_df)
    )

    print(
        "chunk 생성 기사 수:",
        (
            result_df[
                "chunk_count"
            ] > 0
        ).sum()
    )

    print()

    print(
        "평균 chunk 수:",
        round(
            valid_chunk_counts.mean(),
            2,
        )
    )

    print(
        "중앙값:",
        valid_chunk_counts.median()
    )

    print(
        "최대 chunk 수:",
        valid_chunk_counts.max()
    )


    return result_df

In [29]:
hanmi_chunk_df = analyze_case_chunks(
    case_id="hanmi_20260612",
    embedding_model=embedding_model,
)

case_id: hanmi_20260612
company: 한미반도체
전체 기사 수: 1079
chunk 생성 기사 수: 1077

평균 chunk 수: 1.05
중앙값: 1.0
최대 chunk 수: 9


In [30]:
hanmi_chunk_df[
    "chunk_count"
].value_counts(
    sort=False
).sort_index()

chunk_count
0       2
1    1045
2      24
3       4
4       1
7       2
9       1
Name: count, dtype: int64

In [133]:
skhynix_chunk_df = analyze_case_chunks(
    case_id="skhynix_20260304",
    embedding_model=embedding_model,
)

case_id: skhynix_20260304
company: SK하이닉스
전체 기사 수: 18451
chunk 생성 기사 수: 18334

평균 chunk 수: 1.06
중앙값: 1.0
최대 chunk 수: 11


In [134]:
skhynix_chunk_df[
    "chunk_count"
].value_counts(
    sort=False
).sort_index()

chunk_count
0       117
1     17643
2       478
3        95
4        46
5        29
6        24
7        11
8         5
9         2
11        1
Name: count, dtype: int64

In [135]:
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

CHUNK_SIZE = 2048
CHUNK_OVERLAP = 256

In [136]:
def chunk_article_body_with_metadata(
    text: str,
    tokenizer,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
):
    """
    normalized_body를 token 기준으로 chunking합니다.

    각 chunk에 대해:
    - chunk_index
    - start_token
    - end_token
    - token_count
    - unique_token_count
    - text

    를 반환합니다.

    unique_token_count는 overlap으로 이미 앞 chunk에
    포함되었던 token을 제외하고 새롭게 추가되는 token 수입니다.
    """

    if not isinstance(text, str):
        return []

    text = text.strip()

    if not text:
        return []

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap은 chunk_size보다 작아야 합니다."
        )


    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )

    token_ids = encoded[
        "input_ids"
    ]


    if not token_ids:
        return []


    step_size = (
        chunk_size
        - chunk_overlap
    )


    chunks = []

    start = 0
    previous_end = 0
    chunk_index = 0


    while start < len(token_ids):

        end = min(
            start + chunk_size,
            len(token_ids),
        )


        chunk_token_ids = (
            token_ids[
                start:end
            ]
        )


        chunk_text = tokenizer.decode(
            chunk_token_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        ).strip()


        # 첫 chunk는 전체 token이 새로운 내용
        if chunk_index == 0:

            unique_token_count = (
                end - start
            )

        else:

            # 이전 chunk의 end 이후에 새롭게 등장한 token 수
            unique_token_count = max(
                end - previous_end,
                0,
            )


        chunks.append(
            {
                "chunk_index":
                    chunk_index,

                "start_token":
                    start,

                "end_token":
                    end,

                "token_count":
                    end - start,

                "unique_token_count":
                    unique_token_count,

                "text":
                    chunk_text,
            }
        )


        previous_end = end


        if end >= len(token_ids):
            break


        start += step_size
        chunk_index += 1


    return chunks

In [137]:
import numpy as np
import pandas as pd

from datetime import datetime


def create_article_embeddings(
    case_id: str,
    embedding_model,
    embedding_batch_size: int = 4,
    article_batch_size: int = 128,
    overwrite: bool = False,
):
    """
    articles.parquet의 normalized_body를 이용하여
    article embedding을 생성합니다.

    처리 과정:

    normalized_body
        ↓
    token chunking
        ↓
    BGE-M3 chunk embedding
        ↓
    unique_token_count 가중 평균
        ↓
    L2 normalization
        ↓
    article_embeddings.npy
    embedding_metadata.parquet
    """

    paths = get_case_paths(
        case_id
    )


    articles_path = paths[
        "articles"
    ]

    embeddings_path = paths[
        "embeddings"
    ]

    metadata_path = paths[
        "embedding_metadata"
    ]


    # =========================================
    # 기존 파일 확인
    # =========================================

    if not overwrite:

        if (
            embeddings_path.exists()
            or metadata_path.exists()
        ):
            raise FileExistsError(
                "기존 embedding 파일이 존재합니다. "
                "다시 생성하려면 overwrite=True로 실행해주세요."
            )


    # =========================================
    # 데이터 불러오기
    # =========================================

    articles_df = pd.read_parquet(
        articles_path
    )


    required_columns = [
        "article_id",
        "normalized_body",
    ]


    for column in required_columns:

        if column not in articles_df.columns:

            raise KeyError(
                f"필수 컬럼이 없습니다: {column}"
            )


    # =========================================
    # normalized_body 존재 기사만 사용
    # =========================================

    valid_df = (
        articles_df[
            articles_df[
                "normalized_body"
            ].notna()
        ]
        .copy()
    )


    valid_df = (
        valid_df[
            valid_df[
                "normalized_body"
            ]
            .astype(str)
            .str.strip()
            .ne("")
        ]
        .reset_index(drop=True)
    )


    # article_id 중복 검사
    if valid_df[
        "article_id"
    ].duplicated().any():

        raise ValueError(
            "article_id 중복이 존재합니다."
        )


    # =========================================
    # 설정
    # =========================================

    tokenizer = (
        embedding_model.tokenizer
    )


    embedding_dimension = (
        embedding_model
        .get_sentence_embedding_dimension()
    )


    article_count = len(
        valid_df
    )


    article_embeddings = np.empty(
        (
            article_count,
            embedding_dimension,
        ),
        dtype=np.float32,
    )


    metadata_rows = []


    created_at = (
        datetime.now(
            KST
        )
        .isoformat()
    )


    # =========================================
    # Article batch 처리
    # =========================================

    for batch_start in range(
        0,
        article_count,
        article_batch_size,
    ):

        batch_end = min(
            batch_start
            + article_batch_size,
            article_count,
        )


        batch_df = (
            valid_df.iloc[
                batch_start:
                batch_end
            ]
        )


        # 한 batch에 포함된 모든 chunk
        chunk_texts = []

        # article별 chunk 위치
        chunk_positions = {}

        # article별 chunk 가중치
        chunk_weights = {}

        # article별 token/chunk 정보
        article_infos = {}


        # =====================================
        # Chunk 생성
        # =====================================

        for embedding_index, row in (
            batch_df.iterrows()
        ):

            chunks = (
                chunk_article_body_with_metadata(
                    text=row[
                        "normalized_body"
                    ],
                    tokenizer=tokenizer,
                )
            )


            if not chunks:

                raise ValueError(
                    "chunk를 생성할 수 없습니다: "
                    f"{row['article_id']}"
                )


            positions = []
            weights = []


            for chunk in chunks:

                position = len(
                    chunk_texts
                )


                chunk_texts.append(
                    chunk[
                        "text"
                    ]
                )


                positions.append(
                    position
                )


                weights.append(
                    chunk[
                        "unique_token_count"
                    ]
                )


            chunk_positions[
                embedding_index
            ] = positions


            chunk_weights[
                embedding_index
            ] = np.asarray(
                weights,
                dtype=np.float32,
            )


            article_infos[
                embedding_index
            ] = {
                "chunk_count":
                    len(chunks),

                "token_length":
                    int(
                        sum(
                            chunk[
                                "unique_token_count"
                            ]
                            for chunk
                            in chunks
                        )
                    ),
            }


        # =====================================
        # BGE-M3 Chunk Embedding
        # =====================================

        chunk_embeddings = (
            embedding_model.encode(
                chunk_texts,

                batch_size=
                    embedding_batch_size,

                convert_to_numpy=True,

                normalize_embeddings=True,

                show_progress_bar=False,
            )
        )


        chunk_embeddings = (
            np.asarray(
                chunk_embeddings,
                dtype=np.float32,
            )
        )


        # =====================================
        # Article Embedding 생성
        # =====================================

        for embedding_index, row in (
            batch_df.iterrows()
        ):

            positions = (
                chunk_positions[
                    embedding_index
                ]
            )


            weights = (
                chunk_weights[
                    embedding_index
                ]
            )


            embeddings = (
                chunk_embeddings[
                    positions
                ]
            )


            # unique token 수 기반 가중 평균
            article_vector = np.average(
                embeddings,
                axis=0,
                weights=weights,
            )


            # 최종 L2 normalization
            norm = np.linalg.norm(
                article_vector
            )


            if norm == 0:

                raise ValueError(
                    "0-vector embedding이 생성되었습니다: "
                    f"{row['article_id']}"
                )


            article_vector = (
                article_vector
                / norm
            )


            article_embeddings[
                embedding_index
            ] = (
                article_vector.astype(
                    np.float32
                )
            )


            info = article_infos[
                embedding_index
            ]


            metadata_rows.append(
                {
                    "embedding_index":
                        embedding_index,

                    "article_id":
                        row[
                            "article_id"
                        ],

                    "case_id":
                        case_id,

                    "company":
                        CASES[
                            case_id
                        ][
                            "company"
                        ],

                    "embedding_model":
                        EMBEDDING_MODEL_NAME,

                    "embedding_dimension":
                        embedding_dimension,

                    "source_column":
                        "normalized_body",

                    "normalization_method":
                        "whitespace_only",

                    "token_length":
                        info[
                            "token_length"
                        ],

                    "chunk_count":
                        info[
                            "chunk_count"
                        ],

                    "chunk_size":
                        CHUNK_SIZE,

                    "chunk_overlap":
                        CHUNK_OVERLAP,

                    "pooling_method":
                        "unique_token_weighted_mean_l2",

                    "created_at":
                        created_at,
                }
            )


        print(
            f"[{batch_end}/{article_count}] "
            "article embedding 완료"
        )


    # =========================================
    # Metadata 생성
    # =========================================

    metadata_df = pd.DataFrame(
        metadata_rows
    )


    metadata_df = (
        metadata_df
        .sort_values(
            "embedding_index"
        )
        .reset_index(drop=True)
    )


    # =========================================
    # 기본 무결성 검사
    # =========================================

    if len(
        metadata_df
    ) != len(
        article_embeddings
    ):
        raise ValueError(
            "embedding과 metadata 개수가 일치하지 않습니다."
        )


    expected_indices = np.arange(
        len(
            metadata_df
        )
    )


    if not np.array_equal(
        metadata_df[
            "embedding_index"
        ].to_numpy(),
        expected_indices,
    ):
        raise ValueError(
            "embedding_index가 연속적이지 않습니다."
        )


    # =========================================
    # 저장 디렉토리
    # =========================================

    embeddings_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    # =========================================
    # 임시 파일에 먼저 저장
    # =========================================

    temp_embeddings_path = (
        embeddings_path
        .with_suffix(
            ".tmp.npy"
        )
    )


    temp_metadata_path = (
        metadata_path
        .with_suffix(
            ".tmp.parquet"
        )
    )


    np.save(
        temp_embeddings_path,
        article_embeddings,
    )


    metadata_df.to_parquet(
        temp_metadata_path,
        index=False,
    )


    # =========================================
    # 최종 파일로 교체
    # =========================================

    temp_embeddings_path.replace(
        embeddings_path
    )

    temp_metadata_path.replace(
        metadata_path
    )


    # =========================================
    # 결과 출력
    # =========================================

    print()
    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "전체 기사 수:",
        len(
            articles_df
        )
    )

    print(
        "임베딩 생성 기사 수:",
        len(
            metadata_df
        )
    )

    print(
        "임베딩 제외 기사 수:",
        len(
            articles_df
        )
        - len(
            metadata_df
        )
    )

    print(
        "Embedding shape:",
        article_embeddings.shape
    )

    print(
        "Embedding 저장:",
        embeddings_path
    )

    print(
        "Metadata 저장:",
        metadata_path
    )


    return (
        article_embeddings,
        metadata_df,
    )

In [37]:
hanmi_embeddings, hanmi_embedding_metadata = (
    create_article_embeddings(
        case_id="hanmi_20260612",
        embedding_model=embedding_model,
        embedding_batch_size=4,
        article_batch_size=128,
    )
)

/var/folders/h0/q1znprh175l_s9b9pj8xk0cr0000gn/T/ipykernel_81944/2475113907.py:140: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


[128/1077] article embedding 완료
[256/1077] article embedding 완료
[384/1077] article embedding 완료
[512/1077] article embedding 완료
[640/1077] article embedding 완료
[768/1077] article embedding 완료
[896/1077] article embedding 완료
[1024/1077] article embedding 완료
[1077/1077] article embedding 완료

case_id: hanmi_20260612
company: 한미반도체
전체 기사 수: 1079
임베딩 생성 기사 수: 1077
임베딩 제외 기사 수: 2
Embedding shape: (1077, 1024)
Embedding 저장: stored_data/news/hanmi_20260612/article_embeddings.npy
Metadata 저장: stored_data/news/hanmi_20260612/embedding_metadata.parquet


In [38]:
saved_embeddings = np.load(
    get_case_paths(
        "hanmi_20260612"
    )[
        "embeddings"
    ]
)


saved_metadata = pd.read_parquet(
    get_case_paths(
        "hanmi_20260612"
    )[
        "embedding_metadata"
    ]
)


print(
    "Embedding shape:",
    saved_embeddings.shape
)

print(
    "Metadata rows:",
    len(
        saved_metadata
    )
)


assert (
    saved_embeddings.shape[0]
    == len(saved_metadata)
)


assert np.array_equal(
    saved_metadata[
        "embedding_index"
    ].to_numpy(),
    np.arange(
        len(
            saved_metadata
        )
    ),
)


assert not saved_metadata[
    "article_id"
].duplicated().any()


embedding_norms = np.linalg.norm(
    saved_embeddings,
    axis=1,
)


print(
    "Embedding norm 최소:",
    embedding_norms.min()
)

print(
    "Embedding norm 평균:",
    embedding_norms.mean()
)

print(
    "Embedding norm 최대:",
    embedding_norms.max()
)

Embedding shape: (1077, 1024)
Metadata rows: 1077
Embedding norm 최소: 0.9999999
Embedding norm 평균: 1.0
Embedding norm 최대: 1.0000001


In [138]:
skhynix_embeddings, skhynix_embedding_metadata = (
    create_article_embeddings(
        case_id="skhynix_20260304",
        embedding_model=embedding_model,
        embedding_batch_size=4,
        article_batch_size=128,
    )
)

/var/folders/h0/q1znprh175l_s9b9pj8xk0cr0000gn/T/ipykernel_81944/2475113907.py:140: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


[128/18334] article embedding 완료
[256/18334] article embedding 완료
[384/18334] article embedding 완료
[512/18334] article embedding 완료
[640/18334] article embedding 완료
[768/18334] article embedding 완료
[896/18334] article embedding 완료
[1024/18334] article embedding 완료
[1152/18334] article embedding 완료
[1280/18334] article embedding 완료
[1408/18334] article embedding 완료
[1536/18334] article embedding 완료
[1664/18334] article embedding 완료
[1792/18334] article embedding 완료
[1920/18334] article embedding 완료
[2048/18334] article embedding 완료
[2176/18334] article embedding 완료
[2304/18334] article embedding 완료
[2432/18334] article embedding 완료
[2560/18334] article embedding 완료
[2688/18334] article embedding 완료
[2816/18334] article embedding 완료
[2944/18334] article embedding 완료
[3072/18334] article embedding 완료
[3200/18334] article embedding 완료
[3328/18334] article embedding 완료
[3456/18334] article embedding 완료
[3584/18334] article embedding 완료
[3712/18334] article embedding 완료
[3840/18334] article 

In [139]:
saved_embeddings = np.load(
    get_case_paths(
        "skhynix_20260304"
    )[
        "embeddings"
    ]
)


saved_metadata = pd.read_parquet(
    get_case_paths(
        "skhynix_20260304"
    )[
        "embedding_metadata"
    ]
)


print(
    "Embedding shape:",
    saved_embeddings.shape
)

print(
    "Metadata rows:",
    len(
        saved_metadata
    )
)


assert (
    saved_embeddings.shape[0]
    == len(saved_metadata)
)


assert np.array_equal(
    saved_metadata[
        "embedding_index"
    ].to_numpy(),
    np.arange(
        len(
            saved_metadata
        )
    ),
)


assert not saved_metadata[
    "article_id"
].duplicated().any()


embedding_norms = np.linalg.norm(
    saved_embeddings,
    axis=1,
)


print(
    "Embedding norm 최소:",
    embedding_norms.min()
)

print(
    "Embedding norm 평균:",
    embedding_norms.mean()
)

print(
    "Embedding norm 최대:",
    embedding_norms.max()
)

Embedding shape: (18334, 1024)
Metadata rows: 18334
Embedding norm 최소: 0.9999999
Embedding norm 평균: 1.0
Embedding norm 최대: 1.0000001


In [8]:
import numpy as np
import pandas as pd


def to_kst_timestamp(value):
    """
    published_at 값을 KST timezone-aware Timestamp로 변환합니다.

    - timezone이 없는 값은 KST로 간주
    - timezone이 있는 값은 KST로 변환
    """

    if pd.isna(value):
        return pd.NaT

    ts = pd.Timestamp(value)

    if ts.tzinfo is None:
        ts = ts.tz_localize(KST)

    else:
        ts = ts.tz_convert(KST)

    return ts

In [141]:
def get_event_articles(
    case_id: str,
):
    """
    해당 case의 정확한 이벤트 구간에 속하는 기사와
    article embedding을 함께 반환합니다.
    """

    if case_id not in CASES:
        raise ValueError(
            f"등록되지 않은 case_id입니다: {case_id}"
        )

    paths = get_case_paths(
        case_id
    )

    case_info = CASES[
        case_id
    ]

    event_start = case_info[
        "event_start"
    ]

    event_end = case_info[
        "event_end"
    ]


    # =========================================
    # 기사 데이터
    # =========================================

    articles_df = pd.read_parquet(
        paths["articles"]
    )


    if "published_at" not in articles_df.columns:
        raise KeyError(
            "articles.parquet에 published_at 컬럼이 없습니다."
        )


    articles_df[
        "published_at_kst"
    ] = (
        articles_df[
            "published_at"
        ]
        .apply(
            to_kst_timestamp
        )
    )


    # =========================================
    # Embedding metadata
    # =========================================

    metadata_df = pd.read_parquet(
        paths[
            "embedding_metadata"
        ]
    )


    embeddings = np.load(
        paths[
            "embeddings"
        ]
    )


    if len(metadata_df) != len(embeddings):
        raise ValueError(
            "embedding metadata와 embedding 개수가 일치하지 않습니다."
        )


    # =========================================
    # article_id 기준 결합
    # =========================================

    merged_df = articles_df.merge(
        metadata_df[
            [
                "article_id",
                "embedding_index",
            ]
        ],
        on="article_id",
        how="inner",
        validate="one_to_one",
    )


    # =========================================
    # 이벤트 구간 필터
    # =========================================

    event_df = (
        merged_df[
            (
                merged_df[
                    "published_at_kst"
                ]
                >= event_start
            )
            &
            (
                merged_df[
                    "published_at_kst"
                ]
                <= event_end
            )
        ]
        .copy()
        .sort_values(
            [
                "published_at_kst",
                "article_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # 실제 embedding_index 기준으로 추출
    event_embeddings = embeddings[
        event_df[
            "embedding_index"
        ].to_numpy(
            dtype=int
        )
    ]


    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        case_info[
            "company"
        ]
    )

    print(
        "event_start:",
        event_start
    )

    print(
        "event_end:",
        event_end
    )

    print(
        "이벤트 구간 기사 수:",
        len(
            event_df
        )
    )

    print(
        "Event embedding shape:",
        event_embeddings.shape
    )


    return (
        event_df,
        event_embeddings,
    )

In [41]:
hanmi_event_df, hanmi_event_embeddings = (
    get_event_articles(
        "hanmi_20260612"
    )
)

case_id: hanmi_20260612
company: 한미반도체
event_start: 2026-06-11 15:30:00+09:00
event_end: 2026-06-12 15:29:59+09:00
이벤트 구간 기사 수: 103
Event embedding shape: (103, 1024)


In [42]:
hanmi_event_df[
    [
        "article_id",
        "published_at_kst",
        "publisher_name",
        "title",
    ]
]

,article_id,published_at_kst,publisher_name,title
0,A_0c5722b1dffdd06a,2026-06-11 15:37:29+09:00,EBN,[1% 초고수의 선택] '미래에셋증권' 사고 '주성엔지니어링' 팔았다
1,A_7eec052b23850725,2026-06-11 15:42:24+09:00,서울신문,[서울데이터랩]마감 직후 인기 검색 종목 20選
2,A_ecbf6ed34baa7f87,2026-06-11 15:51:34+09:00,아주경제,"한투운용, 반도체 담은 ACE ETF로 수익률 高高"
3,A_ec3c68a7fd19fae5,2026-06-11 15:54:48+09:00,매일일보,[마감시황] 롤러코스터 코스피 7760선 강보합 마감… 코스닥은 4.7% 급등
4,A_e73325cde85fa9de,2026-06-11 16:06:14+09:00,한국경제,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다
...,...,...,...,...
98,A_4f2bcbcb5279d1d2,2026-06-12 14:46:56+09:00,SR타임스,반도체 역대급 호황에…중소형주 관심 커질까
99,A_61ce718cd2d309b0,2026-06-12 14:50:24+09:00,뉴시스,'스페이스X' 상장 앞두고 국장도 들썩…한미반도체 17%↑
100,A_b7b8c345a9aacce2,2026-06-12 15:03:38+09:00,핀포인트뉴스,한미반도체 주가 폭등랠리… HBM 공정용 차폐 장비 수혜 부각
101,A_a89ddfd9af06802c,2026-06-12 15:09:36+09:00,국제뉴스,원익IPS 주가 상한가...반도체 소부장 '급부상'


In [142]:
skhynix_event_df, skhynix_event_embeddings = (
    get_event_articles(
        "skhynix_20260304"
    )
)

case_id: skhynix_20260304
company: SK하이닉스
event_start: 2026-03-03 15:30:00+09:00
event_end: 2026-03-04 15:29:59+09:00
이벤트 구간 기사 수: 880
Event embedding shape: (880, 1024)


In [143]:
skhynix_event_df[
    [
        "article_id",
        "published_at_kst",
        "publisher_name",
        "title",
    ]
]

,article_id,published_at_kst,publisher_name,title
0,A_736f1653537e592f,2026-03-03 15:33:00+09:00,SBS Biz,"코스피, '이란 사태'에 5800선도 붕괴…7.24% 하락 마감"
1,A_c7a11cb6d05a04fb,2026-03-03 15:34:00+09:00,경향신문,"[속보]코스피 7% 급락 마감, 이란 공습 직격탄…환율 26원 급등 1466원으..."
2,A_855fe37f3534e1dc,2026-03-03 15:35:00+09:00,퍼블릭뉴스통신(public news-network for TTL),"KB증권, 신규 고객에 최대 3만원 주식쿠폰 제공 이벤트"
3,A_ae2967bfee6ca254,2026-03-03 15:35:03+09:00,공감신문,삼성전자 7% 하락에 빚투 개미 비명… 반대매매 공포가 짓누르는 코스피...
4,A_1318bd0435ff6871,2026-03-03 15:35:23+09:00,뉴스1,"안교재, 수원시장 출마 선언…""반도체 1시간 생활권 구축"""
...,...,...,...,...
875,A_eb93724bf5bfe9e0,2026-03-04 15:16:00+09:00,데일리안,"구혁채 과기정통부 차관, 첨단 연구장비 국산화 논의"
876,A_f4b3903e78f5f520,2026-03-04 15:22:24+09:00,뉴시스,"'내가 제일 잘 나가' 반도체, 1월 생산 전월대비 4.4%↓…왜?"
877,A_d3ff96ecfd81189f,2026-03-04 15:24:59.538000+09:00,조선비즈,"[마켓뷰] ‘12% 폭락’ 코스피, 이틀 만에 1150포인트 증발… 사상 최대..."
878,A_0cfebb19a93c52aa,2026-03-04 15:26:04+09:00,연합뉴스,"[과학스냅] 고등과학원, 14일 수학 체험형 행사 '파이데이' 개최"


In [3]:
#결정
from sklearn.cluster import AgglomerativeClustering

import numpy as np


CLUSTER_LINKAGE = "average"

SAME_EVENT_SIMILARITY_THRESHOLD = 0.82

In [4]:
def cluster_event_articles(
    case_id: str,
    similarity_threshold: float = SAME_EVENT_SIMILARITY_THRESHOLD,
    save: bool = True,
):
    """
    이벤트 구간 기사들을 BGE-M3 article embedding을 이용해
    동일 사건 단위로 클러스터링합니다.

    - distance metric: cosine
    - linkage: average
    - distance_threshold:
        1 - similarity_threshold

    similarity_threshold는 average-linkage 병합 기준을
    cosine similarity 형태로 표현한 설정값입니다.
    """

    if not (
        0.0
        < similarity_threshold
        < 1.0
    ):
        raise ValueError(
            "similarity_threshold는 "
            "0과 1 사이여야 합니다."
        )


    event_df, event_embeddings = (
        get_event_articles(
            case_id
        )
    )


    article_count = len(
        event_df
    )


    if article_count == 0:
        raise ValueError(
            "이벤트 구간에 임베딩 가능한 "
            "기사가 없습니다."
        )


    # =========================================
    # Clustering
    # =========================================

    if article_count == 1:

        labels = np.array(
            [0],
            dtype=int,
        )

    else:

        distance_threshold = (
            1.0
            - similarity_threshold
        )


        model = AgglomerativeClustering(
            n_clusters=None,
            metric="cosine",
            linkage=CLUSTER_LINKAGE,
            distance_threshold=
                distance_threshold,
        )


        labels = model.fit_predict(
            event_embeddings
        )


    result_df = (
        event_df.copy()
    )


    result_df[
        "_raw_cluster"
    ] = labels


    # =========================================
    # 클러스터 최초 기사 시점 기준으로
    # C_001, C_002 ... 재부여
    # =========================================

    cluster_order = (
        result_df
        .groupby(
            "_raw_cluster",
            as_index=False,
        )
        .agg(
            first_published_at=(
                "published_at_kst",
                "min",
            ),
            first_article_id=(
                "article_id",
                "min",
            ),
        )
        .sort_values(
            [
                "first_published_at",
                "first_article_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    cluster_id_map = {
        row["_raw_cluster"]:
            f"C_{i:03d}"

        for i, (_, row)
        in enumerate(
            cluster_order.iterrows(),
            start=1,
        )
    }


    result_df[
        "cluster_id"
    ] = (
        result_df[
            "_raw_cluster"
        ]
        .map(
            cluster_id_map
        )
    )


    # =========================================
    # 보도 확산량
    # =========================================

    cluster_stats = (
        result_df
        .groupby(
            "cluster_id"
        )
        .agg(
            propagation_count=(
                "article_id",
                "nunique",
            ),

            propagation_source_count=(
                "publisher_name",
                "nunique",
            ),
        )
        .reset_index()
    )


    result_df = result_df.merge(
        cluster_stats,
        on="cluster_id",
        how="left",
        validate="many_to_one",
    )


    # =========================================
    # Clustering 설정 기록
    # =========================================

    result_df[
        "similarity_threshold"
    ] = (
        similarity_threshold
    )

    result_df[
        "clustering_method"
    ] = (
        "agglomerative_average_cosine"
    )


    result_df = (
        result_df
        .drop(
            columns=[
                "_raw_cluster"
            ]
        )
        .sort_values(
            [
                "cluster_id",
                "published_at_kst",
                "article_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # =========================================
    # 저장
    # =========================================

    if save:

        output_path = (
            get_case_paths(
                case_id
            )[
                "case_dir"
            ]
            / "event_clusters.parquet"
        )


        result_df.to_parquet(
            output_path,
            index=False,
        )

    else:

        output_path = None


    # =========================================
    # 요약
    # =========================================

    cluster_sizes = (
        result_df
        .groupby(
            "cluster_id"
        )
        .size()
    )


    print()
    print(
        "case_id:",
        case_id
    )

    print(
        "이벤트 기사 수:",
        len(result_df)
    )

    print(
        "cluster 수:",
        len(cluster_sizes)
    )

    print(
        "singleton cluster 수:",
        int(
            (
                cluster_sizes == 1
            ).sum()
        )
    )

    print(
        "최대 cluster 크기:",
        int(
            cluster_sizes.max()
        )
    )

    print(
        "linkage:",
        CLUSTER_LINKAGE
    )

    print(
        "similarity threshold:",
        similarity_threshold
    )

    if output_path is not None:

        print(
            "저장 경로:",
            output_path
        )


    return result_df

In [56]:
hanmi_clusters = (
    cluster_event_articles(
        case_id="hanmi_20260612"
    )
)

case_id: hanmi_20260612
company: 한미반도체
event_start: 2026-06-11 15:30:00+09:00
event_end: 2026-06-12 15:29:59+09:00
이벤트 구간 기사 수: 103
Event embedding shape: (103, 1024)

case_id: hanmi_20260612
이벤트 기사 수: 103
cluster 수: 42
singleton cluster 수: 31
최대 cluster 크기: 43
linkage: average
similarity threshold: 0.82
저장 경로: stored_data/news/hanmi_20260612/event_clusters.parquet


In [154]:
skhynix_clusters = (
    cluster_event_articles(
        case_id="skhynix_20260304"
    )
)

case_id: skhynix_20260304
company: SK하이닉스
event_start: 2026-03-03 15:30:00+09:00
event_end: 2026-03-04 15:29:59+09:00
이벤트 구간 기사 수: 880
Event embedding shape: (880, 1024)

case_id: skhynix_20260304
이벤트 기사 수: 880
cluster 수: 342
singleton cluster 수: 236
최대 cluster 크기: 74
linkage: average
similarity threshold: 0.82
저장 경로: stored_data/news/skhynix_20260304/event_clusters.parquet


In [155]:
TRUSTED_PUBLISHERS = {
    "연합뉴스",
    "한국경제",
    "매일경제",
    "조선비즈",
    "이데일리",
    "서울경제",
    "전자신문",
    "지디넷코리아",
}

In [156]:
def select_representative_articles(
    case_id: str,
    trusted_publishers=TRUSTED_PUBLISHERS,
    save: bool = True,
):
    """
    각 사건 cluster에서 LLM에 전달할 대표 기사 1개를 선택합니다.

    원칙
    ----
    1. representative_score는 cluster의 전체 기사를 기준으로 계산
    2. 대표 기사 후보는 trusted publisher 기사로 제한
    3. trusted 기사 중 representative_score가 가장 높은 기사 선택
    4. 동점이면 published_at이 빠른 기사 선택
    5. trusted publisher가 없는 cluster는 최종 대표 기사에서 제외

    propagation_count / propagation_source_count는
    trusted 여부와 관계없이 전체 cluster 기준 값을 그대로 사용합니다.
    """

    paths = get_case_paths(
        case_id
    )

    cluster_path = (
        paths["case_dir"]
        / "event_clusters.parquet"
    )


    if not cluster_path.exists():
        raise FileNotFoundError(
            f"event_clusters.parquet을 찾을 수 없습니다: "
            f"{cluster_path}"
        )


    clustered_df = pd.read_parquet(
        cluster_path
    )

    article_embeddings = np.load(
        paths["embeddings"]
    )


    representative_rows = []

    no_trusted_clusters = []


    # =========================================
    # Cluster별 처리
    # =========================================

    for cluster_id, cluster_df in (
        clustered_df.groupby(
            "cluster_id",
            sort=True,
        )
    ):

        cluster_df = (
            cluster_df
            .copy()
            .reset_index(drop=True)
        )


        embedding_indices = (
            cluster_df[
                "embedding_index"
            ]
            .to_numpy(dtype=int)
        )


        cluster_embeddings = (
            article_embeddings[
                embedding_indices
            ]
        )


        # =====================================
        # 전체 cluster 기준 대표성 점수
        # =====================================

        if len(cluster_df) == 1:

            representative_scores = (
                np.array(
                    [1.0],
                    dtype=np.float32,
                )
            )

        else:

            similarity_matrix = (
                cluster_embeddings
                @ cluster_embeddings.T
            )


            representative_scores = (
                (
                    similarity_matrix.sum(axis=1)
                    - 1.0
                )
                / (
                    len(cluster_df)
                    - 1
                )
            )


        cluster_df[
            "representative_score"
        ] = representative_scores


        # =====================================
        # Trusted publisher만 후보
        # =====================================

        trusted_df = (
            cluster_df[
                cluster_df[
                    "publisher_name"
                ].isin(
                    trusted_publishers
                )
            ]
            .copy()
        )


        # trusted 언론사 기사가 없는 cluster
        if trusted_df.empty:

            no_trusted_clusters.append(
                cluster_id
            )

            continue


        # =====================================
        # Trusted 후보 중 대표 기사 선정
        # =====================================

        representative_df = (
            trusted_df
            .sort_values(
                [
                    "representative_score",
                    "published_at_kst",
                    "article_id",
                ],
                ascending=[
                    False,
                    True,
                    True,
                ],
            )
            .head(1)
            .copy()
        )


        representative_df[
            "is_representative"
        ] = True


        representative_rows.append(
            representative_df
        )


    # =========================================
    # 결과
    # =========================================

    if representative_rows:

        representatives_df = (
            pd.concat(
                representative_rows,
                ignore_index=True,
            )
            .sort_values(
                "cluster_id"
            )
            .reset_index(drop=True)
        )

    else:

        representatives_df = pd.DataFrame()


    # =========================================
    # 저장
    # =========================================

    if save:

        output_path = (
            paths["case_dir"]
            / "representative_articles.parquet"
        )


        representatives_df.to_parquet(
            output_path,
            index=False,
        )

    else:

        output_path = None


    # =========================================
    # 출력
    # =========================================

    total_clusters = (
        clustered_df[
            "cluster_id"
        ].nunique()
    )


    print(
        "case_id:",
        case_id
    )

    print(
        "전체 cluster 수:",
        total_clusters
    )

    print(
        "trusted 대표 기사 생성 cluster 수:",
        len(
            representatives_df
        )
    )

    print(
        "trusted 기사 없는 cluster 수:",
        len(
            no_trusted_clusters
        )
    )

    print(
        "trusted 기사 없는 cluster:",
        no_trusted_clusters
    )


    if output_path is not None:

        print(
            "저장 경로:",
            output_path
        )


    return (
        representatives_df,
        no_trusted_clusters,
    )

In [60]:
hanmi_representatives, hanmi_no_trusted_clusters = (
    select_representative_articles(
        case_id="hanmi_20260612"
    )
)

case_id: hanmi_20260612
전체 cluster 수: 42
trusted 대표 기사 생성 cluster 수: 8
trusted 기사 없는 cluster 수: 34
trusted 기사 없는 cluster: ['C_001', 'C_002', 'C_003', 'C_004', 'C_006', 'C_007', 'C_008', 'C_009', 'C_010', 'C_011', 'C_012', 'C_013', 'C_014', 'C_016', 'C_018', 'C_021', 'C_023', 'C_024', 'C_025', 'C_026', 'C_028', 'C_029', 'C_030', 'C_031', 'C_032', 'C_033', 'C_035', 'C_036', 'C_037', 'C_038', 'C_039', 'C_040', 'C_041', 'C_042']
저장 경로: stored_data/news/hanmi_20260612/representative_articles.parquet


In [61]:
hanmi_representatives[
    [
        "cluster_id",
        "publisher_name",
        "published_at_kst",
        "title",
        "representative_score",
        "propagation_count",
        "propagation_source_count",
    ]
].sort_values(
    "propagation_count",
    ascending=False,
)

,cluster_id,publisher_name,published_at_kst,title,representative_score,propagation_count,propagation_source_count
5,C_022,조선비즈,2026-06-12 09:27:40.479000+09:00,"한미반도체, 스페이스X에 500억 베팅…‘테라팹’ 생태계 진입 포석",0.891150,43,40
4,C_020,한국경제,2026-06-12 09:37:23+09:00,"성호전자, 반도체 장비업체 인티맥스 인수…PEF 크레센도와 동맹",0.939024,3,3
6,C_027,서울경제,2026-06-12 10:13:44+09:00,빚투 확산에 내 마통 한도도 줄어드나…고심하는 은행들,0.853212,2,1
0,C_005,한국경제,2026-06-11 16:06:14+09:00,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,1.000000,1,1
1,C_015,서울경제,2026-06-11 17:46:46+09:00,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,1.000000,1,1
2,C_017,한국경제,2026-06-11 18:02:22+09:00,이 변동장에 10% 수익률 낸 상품은 [한경 프리미엄9 TODAY],1.000000,1,1
3,C_019,한국경제,2026-06-12 06:30:03+09:00,"""AI 돈잔치 시작됐다""…'삼전닉스' 투자 확대에 폭등한 종목 [종목+]",1.000000,1,1
7,C_034,서울경제,2026-06-12 14:03:20+09:00,AI 소부장株 들썩…HPSP·원익IPS 나란히 상한가 [이런국장 저런주식],1.000000,1,1


In [157]:
skhynix_representatives, skhynix_no_trusted_clusters = (
    select_representative_articles(
        case_id="skhynix_20260304"
    )
)

case_id: skhynix_20260304
전체 cluster 수: 342
trusted 대표 기사 생성 cluster 수: 68
trusted 기사 없는 cluster 수: 274
trusted 기사 없는 cluster: ['C_002', 'C_004', 'C_009', 'C_010', 'C_011', 'C_012', 'C_014', 'C_015', 'C_016', 'C_017', 'C_018', 'C_019', 'C_020', 'C_021', 'C_022', 'C_023', 'C_024', 'C_026', 'C_027', 'C_028', 'C_029', 'C_030', 'C_032', 'C_033', 'C_034', 'C_035', 'C_036', 'C_037', 'C_038', 'C_039', 'C_041', 'C_043', 'C_044', 'C_045', 'C_046', 'C_047', 'C_049', 'C_050', 'C_054', 'C_055', 'C_057', 'C_060', 'C_062', 'C_063', 'C_065', 'C_067', 'C_068', 'C_069', 'C_070', 'C_071', 'C_072', 'C_074', 'C_075', 'C_076', 'C_077', 'C_079', 'C_080', 'C_081', 'C_082', 'C_083', 'C_084', 'C_085', 'C_086', 'C_087', 'C_088', 'C_089', 'C_090', 'C_091', 'C_092', 'C_093', 'C_094', 'C_095', 'C_096', 'C_097', 'C_098', 'C_099', 'C_100', 'C_101', 'C_103', 'C_104', 'C_106', 'C_107', 'C_108', 'C_110', 'C_111', 'C_112', 'C_114', 'C_115', 'C_116', 'C_117', 'C_118', 'C_121', 'C_124', 'C_125', 'C_126', 'C_127', 'C_128',

In [158]:
skhynix_representatives[
    [
        "cluster_id",
        "publisher_name",
        "published_at_kst",
        "title",
        "representative_score",
        "propagation_count",
        "propagation_source_count",
    ]
].sort_values(
    "propagation_count",
    ascending=False,
)

,cluster_id,publisher_name,published_at_kst,title,representative_score,propagation_count,propagation_source_count
0,C_001,연합뉴스,2026-03-03 16:14:51+09:00,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",0.865723,74,60
4,C_007,한국경제,2026-03-04 09:31:01+09:00,"코스피, '중동 쇼크'에 이틀 연속 매도 사이드카…3%대 급락",0.869868,49,43
51,C_255,이데일리,2026-03-04 10:00:26+09:00,"한화운용 “PLUS ETF, 10조 넘어 100조로…업계 3위권 도약”",0.921290,33,29
6,C_013,연합뉴스,2026-03-03 15:42:28+09:00,[특징주] 이란사태발 코스피 급락에 '20만전자·100만닉스'도 깨져(종합...,0.868025,22,12
11,C_048,연합뉴스,2026-03-04 09:46:14+09:00,"코스피, 중동 위기 확산에 5% 급락 5,500선 내줘…코스닥도 하락(종합)",0.876834,22,20
...,...,...,...,...,...,...,...
37,C_186,매일경제,2026-03-04 07:00:00+09:00,"매경이 전하는 세상의 지식 (매-세-지, 3월 4일)",1.000000,1,1
10,C_042,이데일리,2026-03-03 16:05:40+09:00,SK이노 임협 본격화…노조 “성과급 기준 영업익으로 바꿔야”,1.000000,1,1
42,C_214,한국경제,2026-03-04 08:26:01+09:00,이틀간 '12조' 던진 외국인…조용히 사들인 종목은 [분석+],1.000000,1,1
19,C_064,이데일리,2026-03-03 16:34:16+09:00,HBM이 바꾼 메모리 수출 지형…중국 의존도 확 줄었다,1.000000,1,1


In [159]:
def show_untrusted_clusters(
    case_id: str,
    no_trusted_clusters,
):
    """
    trusted publisher 기사가 없는 cluster들의
    실제 기사 목록을 확인합니다.

    출력:
    - cluster_id
    - propagation_count
    - propagation_source_count
    - published_at
    - publisher
    - title
    - original_link
    """

    paths = get_case_paths(
        case_id
    )

    cluster_path = (
        paths["case_dir"]
        / "event_clusters.parquet"
    )


    if not cluster_path.exists():
        raise FileNotFoundError(
            f"event_clusters.parquet을 찾을 수 없습니다: "
            f"{cluster_path}"
        )


    clustered_df = pd.read_parquet(
        cluster_path
    )


    required_columns = [
        "cluster_id",
        "published_at_kst",
        "publisher_name",
        "title",
        "original_link",
        "propagation_count",
        "propagation_source_count",
    ]


    for column in required_columns:

        if column not in clustered_df.columns:
            raise KeyError(
                f"필수 컬럼이 없습니다: {column}"
            )


    # =========================================
    # trusted 기사 없는 cluster만 추출
    # =========================================

    untrusted_df = (
        clustered_df[
            clustered_df[
                "cluster_id"
            ].isin(
                no_trusted_clusters
            )
        ]
        .copy()
        .sort_values(
            [
                "cluster_id",
                "published_at_kst",
                "publisher_name",
            ]
        )
        .reset_index(drop=True)
    )


    # =========================================
    # Cluster별 출력
    # =========================================

    for cluster_id, cluster_df in (
        untrusted_df.groupby(
            "cluster_id",
            sort=True,
        )
    ):

        propagation_count = int(
            cluster_df[
                "propagation_count"
            ].iloc[0]
        )

        propagation_source_count = int(
            cluster_df[
                "propagation_source_count"
            ].iloc[0]
        )


        print(
            "=" * 120
        )

        print(
            f"{cluster_id}"
            f" | 기사 {propagation_count}개"
            f" | 언론사 {propagation_source_count}개"
        )

        print(
            "=" * 120
        )


        for _, row in cluster_df.iterrows():

            print(
                row[
                    "published_at_kst"
                ],
                "|",
                row[
                    "publisher_name"
                ],
            )

            print(
                "제목:",
                row[
                    "title"
                ]
            )

            print(
                "링크:",
                row[
                    "original_link"
                ]
            )

            print()


    return untrusted_df

In [160]:
def show_trusted_representative_clusters(
    case_id: str,
):
    """
    trusted 대표 기사가 생성된 cluster들의
    전체 기사 목록을 출력합니다.

    출력:
    - cluster_id
    - propagation_count
    - propagation_source_count
    - published_at
    - publisher
    - title
    - original_link
    - 대표 기사 여부
    """

    paths = get_case_paths(
        case_id
    )

    cluster_path = (
        paths["case_dir"]
        / "event_clusters.parquet"
    )

    representative_path = (
        paths["case_dir"]
        / "representative_articles.parquet"
    )


    if not cluster_path.exists():
        raise FileNotFoundError(
            f"event_clusters.parquet을 찾을 수 없습니다: "
            f"{cluster_path}"
        )


    if not representative_path.exists():
        raise FileNotFoundError(
            f"representative_articles.parquet을 찾을 수 없습니다: "
            f"{representative_path}"
        )


    clustered_df = pd.read_parquet(
        cluster_path
    )

    representatives_df = pd.read_parquet(
        representative_path
    )


    representative_cluster_ids = (
        representatives_df[
            "cluster_id"
        ]
        .drop_duplicates()
        .tolist()
    )


    representative_article_ids = set(
        representatives_df[
            "article_id"
        ]
        .tolist()
    )


    # =========================================
    # trusted 대표 기사가 있는 cluster만 추출
    # =========================================

    trusted_cluster_df = (
        clustered_df[
            clustered_df[
                "cluster_id"
            ].isin(
                representative_cluster_ids
            )
        ]
        .copy()
        .sort_values(
            [
                "cluster_id",
                "published_at_kst",
                "publisher_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # =========================================
    # 대표 기사 여부 표시
    # =========================================

    trusted_cluster_df[
        "is_representative"
    ] = (
        trusted_cluster_df[
            "article_id"
        ].isin(
            representative_article_ids
        )
    )


    # =========================================
    # Cluster별 출력
    # =========================================

    for cluster_id, cluster_df in (
        trusted_cluster_df.groupby(
            "cluster_id",
            sort=True,
        )
    ):

        propagation_count = int(
            cluster_df[
                "propagation_count"
            ].iloc[0]
        )

        propagation_source_count = int(
            cluster_df[
                "propagation_source_count"
            ].iloc[0]
        )


        print(
            "=" * 120
        )

        print(
            f"{cluster_id}"
            f" | 기사 {propagation_count}개"
            f" | 언론사 {propagation_source_count}개"
        )

        print(
            "=" * 120
        )


        for _, row in (
            cluster_df.iterrows()
        ):

            if row[
                "is_representative"
            ]:

                print(
                    "[대표 기사]"
                )


            print(
                row[
                    "published_at_kst"
                ],
                "|",
                row[
                    "publisher_name"
                ],
            )

            print(
                "제목:",
                row[
                    "title"
                ]
            )

            print(
                "링크:",
                row[
                    "original_link"
                ]
            )

            print()


    return trusted_cluster_df

In [83]:
hanmi_trusted_cluster_articles = (
    show_trusted_representative_clusters(
        case_id="hanmi_20260612"
    )
)

C_005 | 기사 1개 | 언론사 1개
[대표 기사]
2026-06-11 16:06:14+09:00 | 한국경제
제목: '극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다
링크: https://www.hankyung.com/article/202606110206i

C_015 | 기사 1개 | 언론사 1개
[대표 기사]
2026-06-11 17:46:46+09:00 | 서울경제
제목: 삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF
링크: https://www.sedaily.com/article/20054725?ref=naver

C_017 | 기사 1개 | 언론사 1개
[대표 기사]
2026-06-11 18:02:22+09:00 | 한국경제
제목: 이 변동장에 10% 수익률 낸 상품은 [한경 프리미엄9 TODAY]
링크: https://www.hankyung.com/article/2026061114671

C_019 | 기사 1개 | 언론사 1개
[대표 기사]
2026-06-12 06:30:03+09:00 | 한국경제
제목: "AI 돈잔치 시작됐다"…'삼전닉스' 투자 확대에 폭등한 종목 [종목+]
링크: https://www.hankyung.com/article/2026061112556

C_020 | 기사 3개 | 언론사 3개
2026-06-12 09:02:58+09:00 | 머니투데이
제목: 성호전자, 인티맥스 인수로 반도체 장비 시너지 본격화…크레센도는 주...
링크: https://www.mt.co.kr/stock/2026/06/12/2026061208582169787

[대표 기사]
2026-06-12 09:37:23+09:00 | 한국경제
제목: 성호전자, 반도체 장비업체 인티맥스 인수…PEF 크레센도와 동맹
링크: https://www.hankyung.com/article/202606122357r

2026-06-12 12:05:00+09:00 | 뉴스톱
제목: 성호전자, 핸들러 강자 '인티맥스' 인수...피터 틸의 크레센도 '

In [81]:
hanmi_untrusted_articles = (
    show_untrusted_clusters(
        case_id="hanmi_20260612",
        no_trusted_clusters=
            hanmi_no_trusted_clusters,
    )
)

C_001 | 기사 2개 | 언론사 2개
2026-06-11 15:37:29+09:00 | EBN
제목: [1% 초고수의 선택] '미래에셋증권' 사고 '주성엔지니어링' 팔았다
링크: https://www.ebn.co.kr/news/articleView.html?idxno=1712178

2026-06-11 20:40:00+09:00 | 헤럴드경제
제목: 갈팡질팡 코스피…초고수들은 ‘하닉’ 팔고, ‘이것’ 저가 매수
링크: https://biz.heraldcorp.com/article/10769968?ref=naver

C_002 | 기사 2개 | 언론사 1개
2026-06-11 15:42:24+09:00 | 서울신문
제목: [서울데이터랩]마감 직후 인기 검색 종목 20選
링크: https://www.seoul.co.kr/news/economy/securities/2026/06/11/20260611500187?wlog_tag3=naver

2026-06-12 09:44:18+09:00 | 서울신문
제목: [서울데이터랩]개장 직후 인기 검색 종목 20選
링크: https://www.seoul.co.kr/news/economy/securities/2026/06/12/20260612500053?wlog_tag3=naver

C_003 | 기사 1개 | 언론사 1개
2026-06-11 15:51:34+09:00 | 아주경제
제목: 한투운용, 반도체 담은 ACE ETF로 수익률 高高
링크: https://www.ajunews.com/view/20260611154405843

C_004 | 기사 1개 | 언론사 1개
2026-06-11 15:54:48+09:00 | 매일일보
제목: [마감시황] 롤러코스터 코스피 7760선 강보합 마감… 코스닥은 4.7% 급등
링크: https://www.m-i.kr/news/articleView.html?idxno=1380832

C_006 | 기사 2개 | 언론사 1개
2026-06-11 16:12:16+09:00 | 핀포인트뉴스
제목:

In [161]:
skhynix_trusted_cluster_articles = (
    show_trusted_representative_clusters(
        case_id="skhynix_20260304"
    )
)

C_001 | 기사 74개 | 언론사 60개
2026-03-03 15:33:00+09:00 | SBS Biz
제목: 코스피, '이란 사태'에 5800선도 붕괴…7.24% 하락 마감
링크: https://biz.sbs.co.kr/article_hub/20000294765?division=NAVER

2026-03-03 15:46:11+09:00 | 한국경제
제목: 코스피 6200→5700 한 번에 밀렸다…'이란 쇼크'에 증시 패닉
링크: https://www.hankyung.com/article/2026030360466

2026-03-03 15:49:24+09:00 | TV조선
제목: '검은 화요일' 코스피, 결국 5,800선 내줘…삼성전자·SK하이닉스도 급락
링크: https://news.tvchosun.com/site/data/html_dir/2026/03/03/2026030390191.html

2026-03-03 15:50:34+09:00 | 중앙일보
제목: 코스피 5791.91에 마감...-452.22포인트, 낙폭 역대 최대
링크: https://www.joongang.co.kr/article/25408773

2026-03-03 15:51:00+09:00 | 서울이코노미뉴스
제목: '검은 화요일' 코스피,7.2% 급락 5,800선 내줘…낙폭 역대 최대
링크: http://www.seouleconews.com/news/articleView.html?idxno=89725

2026-03-03 15:53:00+09:00 | 동아일보
제목: 韓증시 아직 못믿나…중동전 터지자 외국인 5조원 ‘썰물’
링크: https://www.donga.com/news/Economy/article/all/20260303/133454926/1

2026-03-03 15:53:25+09:00 | 충남일보
제목: 이란 發 전쟁이 덮은 '검은 화요일'... 코스피 7%대 급락·5800선 붕괴
링크: http://www.chungnamilbo.co.kr/news/articleView

In [162]:
skhynix_untrusted_articles = (
    show_untrusted_clusters(
        case_id="skhynix_20260304",
        no_trusted_clusters=
            skhynix_no_trusted_clusters,
    )
)

C_002 | 기사 3개 | 언론사 3개
2026-03-03 15:34:00+09:00 | 경향신문
제목: [속보]코스피 7% 급락 마감, 이란 공습 직격탄…환율 26원 급등 1466원으...
링크: https://www.khan.co.kr/article/202603031534001

2026-03-03 15:42:26+09:00 | 뉴스퀘스트
제목: '이란 쇼크' 코스피 7% 급락 마감…外人 5조 '패닉 셀'에 사이드카 발동
링크: https://www.newsquest.co.kr/news/articleView.html?idxno=262611

2026-03-03 16:03:00+09:00 | 한경비즈니스
제목: 중동 리스크 어디까지? ‘20만전자·100만닉스’ 붕괴
링크: https://magazine.hankyung.com/business/article/202603036166b

C_004 | 기사 2개 | 언론사 2개
2026-03-03 15:35:03+09:00 | 공감신문
제목: 삼성전자 7% 하락에 빚투 개미 비명… 반대매매 공포가 짓누르는 코스피...
링크: https://www.gokorea.kr/news/articleView.html?idxno=859395

2026-03-04 08:45:00+09:00 | 데일리한국
제목: "상승장 타려고 빚내서 왔는데..." 중동발 악재에 '빚투개미' 초비상
링크: https://daily.hankooki.com/news/articleView.html?idxno=1342079

C_009 | 기사 1개 | 언론사 1개
2026-03-03 15:38:55+09:00 | 헤럴드경제
제목: “외국은 반도체팹 지방 분산, 우리만 수도권 몰빵”
링크: https://biz.heraldcorp.com/article/10685964?ref=naver

C_010 | 기사 3개 | 언론사 3개
2026-03-03 15:41:28+09:00 | THE Biz(더비즈)
제목: [속보] '검은 화요일'...코스피 5800선도

In [5]:
import numpy as np
import pandas as pd


NOVELTY_LOOKBACK_DAYS = 30
FAISS_TOP_K = 50

In [6]:
def get_prior_news_corpus(
    case_id: str,
    lookback_days: int = NOVELTY_LOOKBACK_DAYS,
):
    """
    event_start 이전 lookback_days 기간의
    기사와 article embedding을 반환합니다.

    기간:
    [event_start - lookback_days, event_start)
    """

    if case_id not in CASES:
        raise ValueError(
            f"등록되지 않은 case_id입니다: {case_id}"
        )


    paths = get_case_paths(
        case_id
    )

    event_start = CASES[
        case_id
    ][
        "event_start"
    ]

    prior_start = (
        event_start
        - pd.Timedelta(
            days=lookback_days
        )
    )


    # =========================================
    # 기사
    # =========================================

    articles_df = pd.read_parquet(
        paths["articles"]
    )


    articles_df[
        "published_at_kst"
    ] = (
        articles_df[
            "published_at"
        ]
        .apply(
            to_kst_timestamp
        )
    )


    # =========================================
    # Embedding metadata
    # =========================================

    metadata_df = pd.read_parquet(
        paths[
            "embedding_metadata"
        ]
    )


    embeddings = np.load(
        paths[
            "embeddings"
        ]
    )


    if len(metadata_df) != len(embeddings):
        raise ValueError(
            "embedding metadata와 embedding 개수가 "
            "일치하지 않습니다."
        )


    merged_df = articles_df.merge(
        metadata_df[
            [
                "article_id",
                "embedding_index",
            ]
        ],
        on="article_id",
        how="inner",
        validate="one_to_one",
    )


    # =========================================
    # 이전 30일
    # =========================================

    prior_df = (
        merged_df[
            (
                merged_df[
                    "published_at_kst"
                ]
                >= prior_start
            )
            &
            (
                merged_df[
                    "published_at_kst"
                ]
                < event_start
            )
        ]
        .copy()
        .sort_values(
            [
                "published_at_kst",
                "article_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    prior_embeddings = embeddings[
        prior_df[
            "embedding_index"
        ].to_numpy(
            dtype=int
        )
    ]


    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "prior_start:",
        prior_start
    )

    print(
        "prior_end:",
        event_start
    )

    print(
        "이전 30일 기사 수:",
        len(
            prior_df
        )
    )

    print(
        "Prior embedding shape:",
        prior_embeddings.shape
    )


    return (
        prior_df,
        prior_embeddings,
    )

In [66]:
hanmi_prior_df, hanmi_prior_embeddings = (
    get_prior_news_corpus(
        "hanmi_20260612"
    )
)

case_id: hanmi_20260612
company: 한미반도체
prior_start: 2026-05-12 15:30:00+09:00
prior_end: 2026-06-11 15:30:00+09:00
이전 30일 기사 수: 864
Prior embedding shape: (864, 1024)


In [9]:
skhynix_prior_df, skhynix_prior_embeddings = (
    get_prior_news_corpus(
        "skhynix_20260304"
    )
)

case_id: skhynix_20260304
company: SK하이닉스
prior_start: 2026-02-01 15:30:00+09:00
prior_end: 2026-03-03 15:30:00+09:00
이전 30일 기사 수: 15690
Prior embedding shape: (15690, 1024)


In [10]:
import faiss
import numpy as np
import pandas as pd


def build_faiss_index(
    embeddings: np.ndarray,
):
    """
    L2-normalized article embedding을 이용한
    exact cosine similarity 검색용 FAISS index를 생성합니다.

    article embedding은 이미 L2 normalization되어 있으므로
    inner product = cosine similarity 입니다.
    """

    embeddings = np.asarray(
        embeddings,
        dtype=np.float32,
    )


    if embeddings.ndim != 2:
        raise ValueError(
            "embeddings는 2차원 배열이어야 합니다."
        )


    dimension = embeddings.shape[1]


    index = faiss.IndexFlatIP(
        dimension
    )


    index.add(
        embeddings
    )


    print(
        "FAISS dimension:",
        dimension
    )

    print(
        "FAISS ntotal:",
        index.ntotal
    )


    return index

In [91]:
hanmi_prior_index = build_faiss_index(
    hanmi_prior_embeddings
)

FAISS dimension: 1024
FAISS ntotal: 864


In [11]:
skhynix_prior_index = build_faiss_index(
    skhynix_prior_embeddings
)

FAISS dimension: 1024
FAISS ntotal: 15690


In [12]:
def search_prior_similar_articles(
    case_id: str,
    embedding_model,
    top_k: int = FAISS_TOP_K,
    title_batch_size: int = 32,
):
    """
    trusted 대표 기사 각각에 대해
    event_start 이전 30일 corpus에서
    본문 기반 article embedding cosine similarity로
    Top-K 후보를 검색합니다.

    이후 Top-K 후보에 대해서만
    제목 embedding cosine similarity를 추가 계산합니다.

    저장되는 주요 유사도:
    - body_similarity
    - title_similarity

    Top-K 선정 기준은 body_similarity만 사용합니다.
    """

    paths = get_case_paths(
        case_id
    )


    # =========================================
    # 대표 기사
    # =========================================

    representative_path = (
        paths["case_dir"]
        / "representative_articles.parquet"
    )


    if not representative_path.exists():
        raise FileNotFoundError(
            "representative_articles.parquet을 "
            f"찾을 수 없습니다: {representative_path}"
        )


    representatives_df = pd.read_parquet(
        representative_path
    )


    required_representative_columns = [
        "cluster_id",
        "article_id",
        "publisher_name",
        "title",
        "original_link",
        "embedding_index",
    ]


    for column in required_representative_columns:

        if column not in representatives_df.columns:

            raise KeyError(
                "representative_articles.parquet에 "
                f"필수 컬럼이 없습니다: {column}"
            )


    # =========================================
    # 전체 article embedding
    # =========================================

    all_embeddings = np.load(
        paths["embeddings"]
    )


    # =========================================
    # 이전 30일 corpus
    # =========================================

    prior_df, prior_embeddings = (
        get_prior_news_corpus(
            case_id
        )
    )


    if len(prior_df) == 0:

        raise ValueError(
            "이전 30일 corpus에 기사가 없습니다."
        )


    required_prior_columns = [
        "article_id",
        "published_at_kst",
        "publisher_name",
        "title",
        "original_link",
    ]


    for column in required_prior_columns:

        if column not in prior_df.columns:

            raise KeyError(
                "이전 30일 corpus에 "
                f"필수 컬럼이 없습니다: {column}"
            )


    # =========================================
    # FAISS index
    # =========================================

    index = build_faiss_index(
        prior_embeddings
    )


    actual_top_k = min(
        top_k,
        len(prior_df),
    )


    result_rows = []


    # =========================================
    # 대표 기사별 본문 Top-K 검색
    # =========================================

    for _, representative in (
        representatives_df.iterrows()
    ):

        query_embedding_index = int(
            representative[
                "embedding_index"
            ]
        )


        query_vector = (
            all_embeddings[
                query_embedding_index
            ]
            .astype(
                np.float32
            )
            .reshape(
                1,
                -1,
            )
        )


        similarities, indices = (
            index.search(
                query_vector,
                actual_top_k,
            )
        )


        for rank, (
            body_similarity,
            prior_position,
        ) in enumerate(
            zip(
                similarities[0],
                indices[0],
            ),
            start=1,
        ):

            candidate = (
                prior_df.iloc[
                    prior_position
                ]
            )


            result_rows.append(
                {
                    "cluster_id":
                        representative[
                            "cluster_id"
                        ],

                    # -------------------------
                    # 대표 기사
                    # -------------------------

                    "representative_article_id":
                        representative[
                            "article_id"
                        ],

                    "representative_publisher":
                        representative[
                            "publisher_name"
                        ],

                    "representative_title":
                        representative[
                            "title"
                        ],

                    "representative_original_link":
                        representative[
                            "original_link"
                        ],

                    # -------------------------
                    # 후보 순위 / 본문 유사도
                    # -------------------------

                    "candidate_rank":
                        rank,

                    "body_similarity":
                        float(
                            body_similarity
                        ),

                    # -------------------------
                    # 과거 후보 기사
                    # -------------------------

                    "candidate_article_id":
                        candidate[
                            "article_id"
                        ],

                    "candidate_published_at":
                        candidate[
                            "published_at_kst"
                        ],

                    "candidate_publisher":
                        candidate[
                            "publisher_name"
                        ],

                    "candidate_title":
                        candidate[
                            "title"
                        ],

                    "candidate_original_link":
                        candidate[
                            "original_link"
                        ],
                }
            )


    result_df = pd.DataFrame(
        result_rows
    )


    # =========================================
    # 제목 embedding에 필요한 기사 수집
    # =========================================

    title_map = {}


    for _, row in result_df.iterrows():

        title_map[
            row[
                "representative_article_id"
            ]
        ] = (
            row[
                "representative_title"
            ]
        )


        title_map[
            row[
                "candidate_article_id"
            ]
        ] = (
            row[
                "candidate_title"
            ]
        )


    title_article_ids = list(
        title_map.keys()
    )


    # None / NaN 방어
    titles = []

    for article_id in title_article_ids:

        title = title_map[
            article_id
        ]


        if pd.isna(title):
            title = ""

        else:
            title = str(
                title
            )


        titles.append(
            title
        )


    # =========================================
    # BGE-M3 제목 embedding
    # =========================================

    title_embeddings = (
        embedding_model.encode(
            titles,
            batch_size=
                title_batch_size,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    )


    title_embeddings = np.asarray(
        title_embeddings,
        dtype=np.float32,
    )


    title_embedding_map = {
        article_id:
            title_embeddings[index]

        for index, article_id
        in enumerate(
            title_article_ids
        )
    }


    # =========================================
    # Title cosine similarity
    # =========================================

    title_similarities = []


    for _, row in (
        result_df.iterrows()
    ):

        representative_vector = (
            title_embedding_map[
                row[
                    "representative_article_id"
                ]
            ]
        )


        candidate_vector = (
            title_embedding_map[
                row[
                    "candidate_article_id"
                ]
            ]
        )


        title_similarity = float(
            representative_vector
            @ candidate_vector
        )


        title_similarities.append(
            title_similarity
        )


    result_df[
        "title_similarity"
    ] = (
        title_similarities
    )


    # =========================================
    # 컬럼 순서 정리
    # =========================================

    result_df = result_df[
        [
            "cluster_id",

            "representative_article_id",
            "representative_publisher",
            "representative_title",
            "representative_original_link",

            "candidate_rank",

            "body_similarity",
            "title_similarity",

            "candidate_article_id",
            "candidate_published_at",
            "candidate_publisher",
            "candidate_title",
            "candidate_original_link",
        ]
    ]


    # =========================================
    # 저장
    # =========================================

    output_path = (
        paths["case_dir"]
        / "prior_similarity_candidates.parquet"
    )


    result_df.to_parquet(
        output_path,
        index=False,
    )


    # =========================================
    # 결과 출력
    # =========================================

    print()
    print(
        "case_id:",
        case_id
    )

    print(
        "대표 기사 수:",
        representatives_df[
            "cluster_id"
        ].nunique()
    )

    print(
        "후보 결과 수:",
        len(
            result_df
        )
    )

    print(
        "Top-K:",
        actual_top_k
    )

    print(
        "본문 유사도:",
        "body_similarity"
    )

    print(
        "제목 유사도:",
        "title_similarity"
    )

    print(
        "저장 경로:",
        output_path
    )


    return result_df

In [93]:
hanmi_prior_candidates = (
    search_prior_similar_articles(
        case_id="hanmi_20260612",
        embedding_model=
            embedding_model,
        top_k=50,
    )
)

case_id: hanmi_20260612
company: 한미반도체
prior_start: 2026-05-12 15:30:00+09:00
prior_end: 2026-06-11 15:30:00+09:00
이전 30일 기사 수: 864
Prior embedding shape: (864, 1024)
FAISS dimension: 1024
FAISS ntotal: 864

case_id: hanmi_20260612
대표 기사 수: 8
후보 결과 수: 400
Top-K: 50
본문 유사도: body_similarity
제목 유사도: title_similarity
저장 경로: stored_data/news/hanmi_20260612/prior_similarity_candidates.parquet


In [94]:
hanmi_prior_candidates.columns.tolist()

['cluster_id',
 'representative_article_id',
 'representative_publisher',
 'representative_title',
 'representative_original_link',
 'candidate_rank',
 'body_similarity',
 'title_similarity',
 'candidate_article_id',
 'candidate_published_at',
 'candidate_publisher',
 'candidate_title',
 'candidate_original_link']

In [15]:
skhynix_prior_candidates = (
    search_prior_similar_articles(
        case_id="skhynix_20260304",
        embedding_model=
            embedding_model,
        top_k=50,
    )
)

case_id: skhynix_20260304
company: SK하이닉스
prior_start: 2026-02-01 15:30:00+09:00
prior_end: 2026-03-03 15:30:00+09:00
이전 30일 기사 수: 15690
Prior embedding shape: (15690, 1024)
FAISS dimension: 1024
FAISS ntotal: 15690

case_id: skhynix_20260304
대표 기사 수: 68
후보 결과 수: 3400
Top-K: 50
본문 유사도: body_similarity
제목 유사도: title_similarity
저장 경로: stored_data/news/skhynix_20260304/prior_similarity_candidates.parquet


In [97]:
hanmi_prior_candidates[
    hanmi_prior_candidates[
        "candidate_rank"
    ] <= 5
][
    [
        "cluster_id",
        "representative_title",
        "representative_original_link",
        "candidate_rank",
        "body_similarity",
        "title_similarity",
        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]
].sort_values(
    [
        "cluster_id",
        "candidate_rank",
    ]
)

,cluster_id,representative_title,representative_original_link,candidate_rank,body_similarity,title_similarity,candidate_published_at,candidate_publisher,candidate_title,candidate_original_link
0,C_005,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,https://www.hankyung.com/article/202606110206i,1,0.817527,0.344760,2026-05-27 17:54:48+09:00,매일경제,삼전닉스 2배 ETF는 광풍 … 반도체 소부장·전력株는 찬바람,https://www.mk.co.kr/article/12059111
1,C_005,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,https://www.hankyung.com/article/202606110206i,2,0.813580,0.364805,2026-05-29 14:34:40.681000+09:00,조선비즈,“삼전·닉스만 간다”…단일종목 ETF 4일만에 37조원 싹쓸이,https://biz.chosun.com/stock/stock_general/202...
2,C_005,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,https://www.hankyung.com/article/202606110206i,3,0.767145,0.285963,2026-06-10 17:52:57+09:00,매일경제,美서 반도체 소부장 2배 ETF 나온다,https://www.mk.co.kr/article/12070840
3,C_005,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,https://www.hankyung.com/article/202606110206i,4,0.767014,0.357497,2026-05-29 04:15:00+09:00,머니투데이,[ETF 500조 시대] 'TOP2' 묶어 공격적 투자… '압축형' 포트폴리오 뜬다,https://www.mt.co.kr/stock/2026/05/29/20260528...
4,C_005,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,https://www.hankyung.com/article/202606110206i,5,0.765985,0.437135,2026-05-20 17:12:01+09:00,시사저널이코노미,삼성전자·SK하이닉스 레버리지 ETF로 한미반도체·DB하이텍 주가 폭락?,https://www.sisajournal-e.com/news/articleView...
50,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=n...,1,0.829614,0.599401,2026-05-13 10:02:34+09:00,포쓰저널,"""반도체 TOP2 ETF 경쟁 가열""..삼성운용 'KODEX AI반도체' 리모델링",http://www.4th.kr/news/articleView.html?idxno=...
51,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=n...,2,0.829531,0.625034,2026-05-29 15:12:35+09:00,연합인포맥스,같은 반도체 ETF인데 수익률 '극과 극'…삼전 뛰는데 역주행 레버리지도,https://news.einfomax.co.kr/news/articleView.h...
52,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=n...,3,0.821840,0.599627,2026-05-13 09:26:20+09:00,연합인포맥스,"삼성자산, AI반도체 ETF '투톱 압축' 리모델링…삼전·닉스 비중 50% 확대",https://news.einfomax.co.kr/news/articleView.h...
53,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=n...,4,0.819148,0.614428,2026-05-29 17:06:54+09:00,연합인포맥스,"같은 반도체 ETF, 수익률 '극과 극'…삼전 뛰는데 역주행 레버리지도(종...",https://news.einfomax.co.kr/news/articleView.h...
54,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=n...,5,0.814980,0.568611,2026-05-13 09:12:35+09:00,더페어,"삼성운용, AI 반도체 ETF '리모델링'…삼성전자·SK하이닉스 비중 50%로 ...",https://www.thefairnews.co.kr/news/articleView...


In [16]:
skhynix_prior_candidates[
    skhynix_prior_candidates[
        "candidate_rank"
    ] <= 5
][
    [
        "cluster_id",
        "representative_title",
        "representative_original_link",
        "candidate_rank",
        "body_similarity",
        "title_similarity",
        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]
].sort_values(
    [
        "cluster_id",
        "candidate_rank",
    ]
)

,cluster_id,representative_title,representative_original_link,candidate_rank,body_similarity,title_similarity,candidate_published_at,candidate_publisher,candidate_title,candidate_original_link
0,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,1,0.892357,0.865761,2026-03-03 14:05:00+09:00,한겨레,코스피 7% 급락…역대 최대치 452 낙폭 ‘검은 화요일’,https://www.hani.co.kr/arti/economy/economy_ge...
1,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,2,0.877925,0.486739,2026-03-03 10:07:00+09:00,서울이코노미뉴스,"코스피,美-이란 전쟁에 2%대 내려 6,100대…환율 급등",http://www.seouleconews.com/news/articleView.h...
2,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,3,0.863734,0.443515,2026-03-03 09:50:27+09:00,파이낸스투데이,"코스피, 미·이란 전쟁 여파로 장 초반 6,100대 하락",https://www.fntoday.co.kr/news/articleView.htm...
3,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,4,0.862039,0.466215,2026-03-03 10:05:24+09:00,녹색경제신문,"코스피, 중동발 리스크에 2%대 급락...6100선 내줘",https://www.greened.kr/news/articleView.html?i...
4,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,5,0.857539,0.505058,2026-03-03 11:42:45+09:00,포인트데일리,"코스피, 美·이란 전쟁에 6000대 후퇴…코스닥도 하락",https://www.pointdaily.co.kr/news/articleView....
...,...,...,...,...,...,...,...,...,...,...
3350,C_342,"[과학스냅] 고등과학원, 14일 수학 체험형 행사 '파이데이' 개최",https://www.yna.co.kr/view/AKR2026030412270001...,1,0.580674,0.435458,2026-02-23 01:35:46+09:00,벤처스퀘어,"원티드랩, 직장인 컨퍼런스 ‘하이파이브 2026’ 개최",https://www.venturesquare.net/1040043/
3351,C_342,"[과학스냅] 고등과학원, 14일 수학 체험형 행사 '파이데이' 개최",https://www.yna.co.kr/view/AKR2026030412270001...,2,0.558155,0.374202,2026-02-23 17:48:05+09:00,서울경제,“반도체 수율 혁신 앞장”…세계 최고 학회 가는 K스타트업,https://www.sedaily.com/article/20011596?ref=n...
3352,C_342,"[과학스냅] 고등과학원, 14일 수학 체험형 행사 '파이데이' 개최",https://www.yna.co.kr/view/AKR2026030412270001...,3,0.555406,0.385406,2026-02-11 16:09:25+09:00,중앙이코노미뉴스,[ESG] '반도체'로 꿈을 꾸는 아이들: '하인슈타인',https://www.joongangenews.com/news/articleView...
3353,C_342,"[과학스냅] 고등과학원, 14일 수학 체험형 행사 '파이데이' 개최",https://www.yna.co.kr/view/AKR2026030412270001...,4,0.553482,0.279400,2026-02-13 16:30:30+09:00,시사오늘,"한국투자증권 인니법인, 최우수 로컬 통화 채권상 수상…삼성증권, 설 ...",http://www.sisaon.co.kr/news/articleView.html?...


In [17]:
def show_prior_candidates(
    candidates_df,
    top_n: int = 5,
):
    """
    대표 기사별 과거 유사 후보를 출력합니다.

    표시:
    - 대표 기사 제목 / 링크
    - candidate rank
    - body similarity
    - title similarity
    - 과거 기사 시각
    - 언론사
    - 제목
    - 링크
    """

    cluster_ids = (
        candidates_df[
            "cluster_id"
        ]
        .drop_duplicates()
        .tolist()
    )


    for cluster_id in cluster_ids:

        cluster_df = (
            candidates_df[
                candidates_df[
                    "cluster_id"
                ] == cluster_id
            ]
            .sort_values(
                "candidate_rank"
            )
            .head(
                top_n
            )
        )


        representative_title = (
            cluster_df[
                "representative_title"
            ]
            .iloc[0]
        )


        representative_link = (
            cluster_df[
                "representative_original_link"
            ]
            .iloc[0]
        )


        print(
            "=" * 120
        )

        print(
            cluster_id
        )

        print(
            "대표 기사:",
            representative_title
        )

        print(
            "대표 기사 링크:",
            representative_link
        )

        print(
            "=" * 120
        )


        for _, row in (
            cluster_df.iterrows()
        ):

            print(
                f"[{int(row['candidate_rank'])}] "
                f"body={row['body_similarity']:.4f} | "
                f"title={row['title_similarity']:.4f}"
            )


            print(
                row[
                    "candidate_published_at"
                ],
                "|",
                row[
                    "candidate_publisher"
                ],
            )


            print(
                "제목:",
                row[
                    "candidate_title"
                ]
            )


            print(
                "링크:",
                row[
                    "candidate_original_link"
                ]
            )


            print()

In [99]:
show_prior_candidates(
    hanmi_prior_candidates,
    top_n=5,
)

C_005
대표 기사: '극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다
대표 기사 링크: https://www.hankyung.com/article/202606110206i
[1] body=0.8175 | title=0.3448
2026-05-27 17:54:48+09:00 | 매일경제
제목: 삼전닉스 2배 ETF는 광풍 … 반도체 소부장·전력株는 찬바람
링크: https://www.mk.co.kr/article/12059111

[2] body=0.8136 | title=0.3648
2026-05-29 14:34:40.681000+09:00 | 조선비즈
제목: “삼전·닉스만 간다”…단일종목 ETF 4일만에 37조원 싹쓸이
링크: https://biz.chosun.com/stock/stock_general/2026/06/03/KNBFNFKEUFD5RPLOE2UM4QMABA/?utm_source=naver&utm_medium=original&utm_campaign=biz

[3] body=0.7671 | title=0.2860
2026-06-10 17:52:57+09:00 | 매일경제
제목: 美서 반도체 소부장 2배 ETF 나온다
링크: https://www.mk.co.kr/article/12070840

[4] body=0.7670 | title=0.3575
2026-05-29 04:15:00+09:00 | 머니투데이
제목: [ETF 500조 시대] 'TOP2' 묶어 공격적 투자… '압축형' 포트폴리오 뜬다
링크: https://www.mt.co.kr/stock/2026/05/29/2026052820135459730

[5] body=0.7660 | title=0.4371
2026-05-20 17:12:01+09:00 | 시사저널이코노미
제목: 삼성전자·SK하이닉스 레버리지 ETF로 한미반도체·DB하이텍 주가 폭락?
링크: https://www.sisajournal-e.com/news/articleView.html?idxno=421204

C_

In [18]:
show_prior_candidates(
    skhynix_prior_candidates,
    top_n=5,
)

C_001
대표 기사: '검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)
대표 기사 링크: https://www.yna.co.kr/view/AKR20260303134800008?input=1195m
[1] body=0.8924 | title=0.8658
2026-03-03 14:05:00+09:00 | 한겨레
제목: 코스피 7% 급락…역대 최대치 452 낙폭 ‘검은 화요일’
링크: https://www.hani.co.kr/arti/economy/economy_general/1247390.html

[2] body=0.8779 | title=0.4867
2026-03-03 10:07:00+09:00 | 서울이코노미뉴스
제목: 코스피,美-이란 전쟁에 2%대 내려 6,100대…환율 급등
링크: http://www.seouleconews.com/news/articleView.html?idxno=89700

[3] body=0.8637 | title=0.4435
2026-03-03 09:50:27+09:00 | 파이낸스투데이
제목: 코스피, 미·이란 전쟁 여파로 장 초반 6,100대 하락
링크: https://www.fntoday.co.kr/news/articleView.html?idxno=378657

[4] body=0.8620 | title=0.4662
2026-03-03 10:05:24+09:00 | 녹색경제신문
제목: 코스피, 중동발 리스크에 2%대 급락...6100선 내줘
링크: https://www.greened.kr/news/articleView.html?idxno=337492

[5] body=0.8575 | title=0.5051
2026-03-03 11:42:45+09:00 | 포인트데일리
제목: 코스피, 美·이란 전쟁에 6000대 후퇴…코스닥도 하락
링크: https://www.pointdaily.co.kr/news/articleView.html?idxno=294708

C_003
대표 기사: KB證, 비대면 신규고객 대

In [19]:
NOVELTY_BODY_THRESHOLD = 0.82
NOVELTY_TITLE_THRESHOLD = 0.55

In [104]:
hanmi_repeated_candidates = (
    hanmi_prior_candidates[
        (
            hanmi_prior_candidates[
                "body_similarity"
            ]
            >= NOVELTY_BODY_THRESHOLD
        )
        &
        (
            hanmi_prior_candidates[
                "title_similarity"
            ]
            >= NOVELTY_TITLE_THRESHOLD
        )
    ]
    .copy()
)

In [20]:
skhynix_repeated_candidates = (
    skhynix_prior_candidates[
        (
            skhynix_prior_candidates[
                "body_similarity"
            ]
            >= NOVELTY_BODY_THRESHOLD
        )
        &
        (
            skhynix_prior_candidates[
                "title_similarity"
            ]
            >= NOVELTY_TITLE_THRESHOLD
        )
    ]
    .copy()
)

In [108]:
pd.set_option(
    "display.max_colwidth",
    None,
)

In [109]:
hanmi_repeated_candidates[
    [
        "cluster_id",
        "representative_title",
        "representative_original_link",
        "candidate_rank",
        "body_similarity",
        "title_similarity",

        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]
].sort_values(
    [
        "cluster_id",
        "candidate_published_at",
    ]
)

,cluster_id,representative_title,representative_original_link,candidate_rank,body_similarity,title_similarity,candidate_published_at,candidate_publisher,candidate_title,candidate_original_link
52,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,3,0.821840,0.599627,2026-05-13 09:26:20+09:00,연합인포맥스,"삼성자산, AI반도체 ETF '투톱 압축' 리모델링…삼전·닉스 비중 50% 확대",https://news.einfomax.co.kr/news/articleView.html?idxno=4414509
50,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,1,0.829614,0.599401,2026-05-13 10:02:34+09:00,포쓰저널,"""반도체 TOP2 ETF 경쟁 가열""..삼성운용 'KODEX AI반도체' 리모델링",http://www.4th.kr/news/articleView.html?idxno=2111671
51,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,2,0.829531,0.625034,2026-05-29 15:12:35+09:00,연합인포맥스,같은 반도체 ETF인데 수익률 '극과 극'…삼전 뛰는데 역주행 레버리지도,https://news.einfomax.co.kr/news/articleView.html?idxno=4417495


In [21]:
skhynix_repeated_candidates[
    [
        "cluster_id",
        "representative_title",
        "representative_original_link",
        "candidate_rank",
        "body_similarity",
        "title_similarity",

        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]
].sort_values(
    [
        "cluster_id",
        "candidate_published_at",
    ]
)

,cluster_id,representative_title,representative_original_link,candidate_rank,body_similarity,title_similarity,candidate_published_at,candidate_publisher,candidate_title,candidate_original_link
12,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,13,0.839264,0.735829,2026-03-03 09:20:55+09:00,뉴데일리,"코스피 '매도 사이드카', 7% 폭락 5800마저 붕괴 … 확전에 韓 증시 '검은...",https://biz.newdaily.co.kr/site/data/html/2026...
10,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,11,0.841772,0.739952,2026-03-03 11:47:01+09:00,중앙일보,"코스피 7% 이상 급락, 5800선도 깨졌다…외국인 5조 넘게 ‘투매’",https://www.joongang.co.kr/article/25408719
17,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,18,0.834843,0.565551,2026-03-03 13:17:07+09:00,매일신문,"'美-이란 전쟁 쇼크' 코스피, 급락에 매도 사이드카 발동…장중 5900선도...",https://www.imaeil.com/page/view/2026030313140...
0,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,1,0.892357,0.865761,2026-03-03 14:05:00+09:00,한겨레,코스피 7% 급락…역대 최대치 452 낙폭 ‘검은 화요일’,https://www.hani.co.kr/arti/economy/economy_ge...
24,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,25,0.827594,0.577687,2026-03-03 14:09:45+09:00,오피니언뉴스,"코스피, 장 중 5900선 하회...20만전자·100만닉스도 무너져",http://www.opinionnews.co.kr/news/articleView....
...,...,...,...,...,...,...,...,...,...,...
3100,C_298,"바닥인 줄 알고 풀베팅했는데…""나 떨고 있니?"" 개미들 비명",https://www.hankyung.com/article/202603048289i,1,0.825861,0.608575,2026-02-03 17:13:35+09:00,한국경제,급락장서 레버리지에 베팅…간 큰 '전투개미' 웃었다,https://www.hankyung.com/article/2026020315671
3201,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,2,0.835166,0.574641,2026-03-03 13:11:44+09:00,한국경제,'이란 쇼크'에 코스피 매도 사이드카…'20만전자' 위태,https://www.hankyung.com/article/2026030356316
3202,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,3,0.824786,0.569476,2026-03-03 13:52:18+09:00,뉴스퀘스트,중동발 '검은 화요일'…코스피 사이드카 발동·삼성전자 20만원 깨져,https://www.newsquest.co.kr/news/articleView.h...
3204,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,5,0.820939,0.699588,2026-03-03 14:22:31+09:00,연합뉴스,[특징주] 이란 사태에 삼성전자 20만원·SK하이닉스 100만원 깨져,https://www.yna.co.kr/view/AKR2026030311050000...


In [22]:
import pandas as pd


NOVELTY_BODY_THRESHOLD = 0.82
NOVELTY_TITLE_THRESHOLD = 0.55
NOVELTY_TOP_K = 50


def build_news_novelty(
    case_id: str,
    body_threshold: float = NOVELTY_BODY_THRESHOLD,
    title_threshold: float = NOVELTY_TITLE_THRESHOLD,
):
    """
    prior_similarity_candidates.parquet의 Top-K 후보를 이용해
    대표 기사별 NEW / REPEATED를 판정합니다.

    REPEATED 조건:
        body_similarity >= body_threshold
        AND
        title_similarity >= title_threshold

    earliest_similar_date:
        Top-K 후보 중 위 조건을 만족한 기사 가운데
        가장 이른 기사 날짜

    저장:
        news_novelty.parquet
        novelty_matches.parquet
    """

    paths = get_case_paths(
        case_id
    )

    candidates_path = (
        paths["case_dir"]
        / "prior_similarity_candidates.parquet"
    )

    representatives_path = (
        paths["case_dir"]
        / "representative_articles.parquet"
    )


    # =========================================
    # 파일 확인
    # =========================================

    if not candidates_path.exists():
        raise FileNotFoundError(
            "prior_similarity_candidates.parquet을 "
            f"찾을 수 없습니다: {candidates_path}"
        )

    if not representatives_path.exists():
        raise FileNotFoundError(
            "representative_articles.parquet을 "
            f"찾을 수 없습니다: {representatives_path}"
        )


    # =========================================
    # 데이터 로드
    # =========================================

    candidates_df = pd.read_parquet(
        candidates_path
    )

    representatives_df = pd.read_parquet(
        representatives_path
    )


    required_columns = [
        "cluster_id",
        "representative_article_id",
        "representative_publisher",
        "representative_title",
        "representative_original_link",

        "candidate_rank",
        "body_similarity",
        "title_similarity",

        "candidate_article_id",
        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]


    for column in required_columns:

        if column not in candidates_df.columns:
            raise KeyError(
                f"필수 컬럼이 없습니다: {column}"
            )


    # =========================================
    # 후보 날짜 타입 통일
    # =========================================

    candidates_df[
        "candidate_published_at"
    ] = pd.to_datetime(
        candidates_df[
            "candidate_published_at"
        ]
    )


    # =========================================
    # 동일/유사 사건 조건
    # =========================================

    candidates_df[
        "is_similar_event"
    ] = (
        (
            candidates_df[
                "body_similarity"
            ]
            >= body_threshold
        )
        &
        (
            candidates_df[
                "title_similarity"
            ]
            >= title_threshold
        )
    )


    # 판정 조건을 만족한 후보만
    matches_df = (
        candidates_df[
            candidates_df[
                "is_similar_event"
            ]
        ]
        .copy()
        .sort_values(
            [
                "cluster_id",
                "candidate_published_at",
                "candidate_rank",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # =========================================
    # 대표 기사별 최종 판정
    # =========================================

    novelty_rows = []


    for _, representative in (
        representatives_df.iterrows()
    ):

        cluster_id = (
            representative[
                "cluster_id"
            ]
        )


        cluster_matches = (
            matches_df[
                matches_df[
                    "cluster_id"
                ] == cluster_id
            ]
            .copy()
        )


        # =====================================
        # NEW
        # =====================================

        if cluster_matches.empty:

            novelty_rows.append(
                {
                    "cluster_id":
                        cluster_id,

                    "representative_article_id":
                        representative[
                            "article_id"
                        ],

                    "representative_publisher":
                        representative[
                            "publisher_name"
                        ],

                    "representative_title":
                        representative[
                            "title"
                        ],

                    "representative_original_link":
                        representative[
                            "original_link"
                        ],

                    "novelty_status":
                        "NEW",

                    "earliest_similar_date":
                        None,

                    "matched_article_count":
                        0,

                    "best_match_article_id":
                        None,

                    "best_match_publisher":
                        None,

                    "best_match_title":
                        None,

                    "best_match_original_link":
                        None,

                    "best_match_body_similarity":
                        None,

                    "best_match_title_similarity":
                        None,

                    "body_threshold":
                        body_threshold,

                    "title_threshold":
                        title_threshold,

                    "search_top_k":
                        NOVELTY_TOP_K,
                }
            )


        # =====================================
        # REPEATED
        # =====================================

        else:

            # 가장 이른 유사 기사
            earliest_match = (
                cluster_matches
                .sort_values(
                    [
                        "candidate_published_at",
                        "candidate_rank",
                    ]
                )
                .iloc[0]
            )


            earliest_similar_date = (
                earliest_match[
                    "candidate_published_at"
                ]
                .date()
                .isoformat()
            )


            # 가장 강한 match
            # 본문 유사도를 우선,
            # 동률이면 제목 유사도로 결정
            best_match = (
                cluster_matches
                .sort_values(
                    [
                        "body_similarity",
                        "title_similarity",
                        "candidate_rank",
                    ],
                    ascending=[
                        False,
                        False,
                        True,
                    ],
                )
                .iloc[0]
            )


            novelty_rows.append(
                {
                    "cluster_id":
                        cluster_id,

                    "representative_article_id":
                        representative[
                            "article_id"
                        ],

                    "representative_publisher":
                        representative[
                            "publisher_name"
                        ],

                    "representative_title":
                        representative[
                            "title"
                        ],

                    "representative_original_link":
                        representative[
                            "original_link"
                        ],

                    "novelty_status":
                        "REPEATED",

                    "earliest_similar_date":
                        earliest_similar_date,

                    "matched_article_count":
                        cluster_matches[
                            "candidate_article_id"
                        ].nunique(),

                    "best_match_article_id":
                        best_match[
                            "candidate_article_id"
                        ],

                    "best_match_publisher":
                        best_match[
                            "candidate_publisher"
                        ],

                    "best_match_title":
                        best_match[
                            "candidate_title"
                        ],

                    "best_match_original_link":
                        best_match[
                            "candidate_original_link"
                        ],

                    "best_match_body_similarity":
                        float(
                            best_match[
                                "body_similarity"
                            ]
                        ),

                    "best_match_title_similarity":
                        float(
                            best_match[
                                "title_similarity"
                            ]
                        ),

                    "body_threshold":
                        body_threshold,

                    "title_threshold":
                        title_threshold,

                    "search_top_k":
                        NOVELTY_TOP_K,
                }
            )


    # =========================================
    # DataFrame 생성
    # =========================================

    novelty_df = pd.DataFrame(
        novelty_rows
    )


    novelty_df = (
        novelty_df
        .sort_values(
            "cluster_id"
        )
        .reset_index(
            drop=True
        )
    )


    # =========================================
    # 저장
    # =========================================

    novelty_path = (
        paths["case_dir"]
        / "news_novelty.parquet"
    )

    matches_path = (
        paths["case_dir"]
        / "novelty_matches.parquet"
    )


    novelty_df.to_parquet(
        novelty_path,
        index=False,
    )


    matches_df.to_parquet(
        matches_path,
        index=False,
    )


    # =========================================
    # 결과 출력
    # =========================================

    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "대표 기사 수:",
        len(
            novelty_df
        )
    )

    print(
        "NEW:",
        int(
            (
                novelty_df[
                    "novelty_status"
                ]
                == "NEW"
            ).sum()
        )
    )

    print(
        "REPEATED:",
        int(
            (
                novelty_df[
                    "novelty_status"
                ]
                == "REPEATED"
            ).sum()
        )
    )

    print(
        "유사 판정 근거 기사 수:",
        len(
            matches_df
        )
    )

    print(
        "novelty 저장:",
        novelty_path
    )

    print(
        "match 근거 저장:",
        matches_path
    )


    return (
        novelty_df,
        matches_df,
    )

In [111]:
hanmi_novelty, hanmi_novelty_matches = (
    build_news_novelty(
        case_id="hanmi_20260612"
    )
)

case_id: hanmi_20260612
company: 한미반도체
대표 기사 수: 8
NEW: 7
REPEATED: 1
유사 판정 근거 기사 수: 3
novelty 저장: stored_data/news/hanmi_20260612/news_novelty.parquet
match 근거 저장: stored_data/news/hanmi_20260612/novelty_matches.parquet


In [23]:
skhynix_novelty, skhynix_novelty_matches = (
    build_news_novelty(
        case_id="skhynix_20260304"
    )
)

case_id: skhynix_20260304
company: SK하이닉스
대표 기사 수: 68
NEW: 36
REPEATED: 32
유사 판정 근거 기사 수: 229
novelty 저장: stored_data/news/skhynix_20260304/news_novelty.parquet
match 근거 저장: stored_data/news/skhynix_20260304/novelty_matches.parquet


In [112]:
hanmi_novelty[
    [
        "cluster_id",

        "representative_title",
        "representative_original_link",

        "novelty_status",
        "earliest_similar_date",
        "matched_article_count",

        "best_match_publisher",
        "best_match_title",
        "best_match_original_link",

        "best_match_body_similarity",
        "best_match_title_similarity",
    ]
]

,cluster_id,representative_title,representative_original_link,novelty_status,earliest_similar_date,matched_article_count,best_match_publisher,best_match_title,best_match_original_link,best_match_body_similarity,best_match_title_similarity
0,C_005,'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다,https://www.hankyung.com/article/202606110206i,NEW,None,0,None,None,None,NaN,NaN
1,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,REPEATED,2026-05-13,3,포쓰저널,"""반도체 TOP2 ETF 경쟁 가열""..삼성운용 'KODEX AI반도체' 리모델링",http://www.4th.kr/news/articleView.html?idxno=2111671,0.829614,0.599401
2,C_017,이 변동장에 10% 수익률 낸 상품은 [한경 프리미엄9 TODAY],https://www.hankyung.com/article/2026061114671,NEW,None,0,None,None,None,NaN,NaN
3,C_019,"""AI 돈잔치 시작됐다""…'삼전닉스' 투자 확대에 폭등한 종목 [종목+]",https://www.hankyung.com/article/2026061112556,NEW,None,0,None,None,None,NaN,NaN
4,C_020,"성호전자, 반도체 장비업체 인티맥스 인수…PEF 크레센도와 동맹",https://www.hankyung.com/article/202606122357r,NEW,None,0,None,None,None,NaN,NaN
5,C_022,"한미반도체, 스페이스X에 500억 베팅…‘테라팹’ 생태계 진입 포석",https://biz.chosun.com/it-science/ict/2026/06/12/SD65BXVALRDIJLA674VGWGRUQY/?utm_source=naver&utm_medium=original&utm_campaign=biz,NEW,None,0,None,None,None,NaN,NaN
6,C_027,빚투 확산에 내 마통 한도도 줄어드나…고심하는 은행들,https://www.sedaily.com/article/20054947?ref=naver,NEW,None,0,None,None,None,NaN,NaN
7,C_034,AI 소부장株 들썩…HPSP·원익IPS 나란히 상한가 [이런국장 저런주식],https://www.sedaily.com/article/20055067?ref=naver,NEW,None,0,None,None,None,NaN,NaN


In [113]:
hanmi_novelty_matches[
    [
        "cluster_id",

        "representative_title",
        "representative_original_link",

        "candidate_rank",
        "body_similarity",
        "title_similarity",

        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]
].sort_values(
    [
        "cluster_id",
        "candidate_published_at",
    ]
)

,cluster_id,representative_title,representative_original_link,candidate_rank,body_similarity,title_similarity,candidate_published_at,candidate_publisher,candidate_title,candidate_original_link
0,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,3,0.821840,0.599627,2026-05-13 09:26:20+09:00,연합인포맥스,"삼성자산, AI반도체 ETF '투톱 압축' 리모델링…삼전·닉스 비중 50% 확대",https://news.einfomax.co.kr/news/articleView.html?idxno=4414509
1,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,1,0.829614,0.599401,2026-05-13 10:02:34+09:00,포쓰저널,"""반도체 TOP2 ETF 경쟁 가열""..삼성운용 'KODEX AI반도체' 리모델링",http://www.4th.kr/news/articleView.html?idxno=2111671
2,C_015,삼성전기·한미반도체 널뛰기…반도체 장비株 흔든 ETF,https://www.sedaily.com/article/20054725?ref=naver,2,0.829531,0.625034,2026-05-29 15:12:35+09:00,연합인포맥스,같은 반도체 ETF인데 수익률 '극과 극'…삼전 뛰는데 역주행 레버리지도,https://news.einfomax.co.kr/news/articleView.html?idxno=4417495


In [24]:
skhynix_novelty[
    [
        "cluster_id",

        "representative_title",
        "representative_original_link",

        "novelty_status",
        "earliest_similar_date",
        "matched_article_count",

        "best_match_publisher",
        "best_match_title",
        "best_match_original_link",

        "best_match_body_similarity",
        "best_match_title_similarity",
    ]
]

,cluster_id,representative_title,representative_original_link,novelty_status,earliest_similar_date,matched_article_count,best_match_publisher,best_match_title,best_match_original_link,best_match_body_similarity,best_match_title_similarity
0,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,REPEATED,2026-03-03,7,한겨레,코스피 7% 급락…역대 최대치 452 낙폭 ‘검은 화요일’,https://www.hani.co.kr/arti/economy/economy_ge...,0.892357,0.865761
1,C_003,"KB證, 비대면 신규고객 대상 이벤트",https://www.mk.co.kr/article/11977548,REPEATED,2026-03-03,29,뉴스로드,"KB증권, 비대면 첫 계좌에 '주식쿠폰·우량주·평생 수수료' 모두 푼다",http://www.newsroad.co.kr/news/articleView.htm...,0.900357,0.704229
2,C_005,"안교재, 수원시장 출마…반도체 벨트 1시간 생활권·산업교통망 공약",https://www.etnews.com/20260303000587,REPEATED,2026-03-03,4,천지일보,"안교재 예비후보, 수원시장 선거 출마 공식 선언",https://www.newscj.com/news/articleView.html?i...,0.925177,0.751875
3,C_006,"코스피, 7% 급락…개인투자자 '저가 매수' 노렸다",https://www.etnews.com/20260303000470,REPEATED,2026-03-03,9,더팩트,중동 리스크에 코스피 6000선 붕괴…삼성전자·SK하이닉스 6%대 급락,https://news.tf.co.kr/read/economy/2297901.htm,0.872000,0.557396
4,C_007,"코스피, '중동 쇼크'에 이틀 연속 매도 사이드카…3%대 급락",https://www.hankyung.com/article/2026030474886,REPEATED,2026-02-02,42,에너지경제,"코스피 4%대 급락, 외국인 ‘팔자’·개인 ‘사자’…환율 1470원 턱밑 추...",https://www.ekn.kr/web/view.php?key=2026030302...,0.901603,0.557037
...,...,...,...,...,...,...,...,...,...,...,...
63,C_302,"SK하이닉스 산 블랙록, 이수페타시스는 차익실현했다 [매경 자이앤트]",https://www.mk.co.kr/article/11978354,NEW,None,0,None,None,None,NaN,NaN
64,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,REPEATED,2026-03-03,4,한국경제,'이란 쇼크'에 코스피 매도 사이드카…'20만전자' 위태,https://www.hankyung.com/article/2026030356316,0.835166,0.574641
65,C_330,"울산폴리텍대학, ""전국 41개 폴리텍 대학에서 취업률 1위 기록""",https://www.hankyung.com/article/202603048559h,NEW,None,0,None,None,None,NaN,NaN
66,C_335,"경기지사 출마 與 한준호 ""'경기반도체학교' 만들겠다"" [6·3 지방선거]",https://www.hankyung.com/article/202603012082i,NEW,None,0,None,None,None,NaN,NaN


In [25]:
skhynix_novelty_matches[
    [
        "cluster_id",

        "representative_title",
        "representative_original_link",

        "candidate_rank",
        "body_similarity",
        "title_similarity",

        "candidate_published_at",
        "candidate_publisher",
        "candidate_title",
        "candidate_original_link",
    ]
].sort_values(
    [
        "cluster_id",
        "candidate_published_at",
    ]
)

,cluster_id,representative_title,representative_original_link,candidate_rank,body_similarity,title_similarity,candidate_published_at,candidate_publisher,candidate_title,candidate_original_link
0,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,13,0.839264,0.735829,2026-03-03 09:20:55+09:00,뉴데일리,"코스피 '매도 사이드카', 7% 폭락 5800마저 붕괴 … 확전에 韓 증시 '검은...",https://biz.newdaily.co.kr/site/data/html/2026...
1,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,11,0.841772,0.739952,2026-03-03 11:47:01+09:00,중앙일보,"코스피 7% 이상 급락, 5800선도 깨졌다…외국인 5조 넘게 ‘투매’",https://www.joongang.co.kr/article/25408719
2,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,18,0.834843,0.565551,2026-03-03 13:17:07+09:00,매일신문,"'美-이란 전쟁 쇼크' 코스피, 급락에 매도 사이드카 발동…장중 5900선도...",https://www.imaeil.com/page/view/2026030313140...
3,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,1,0.892357,0.865761,2026-03-03 14:05:00+09:00,한겨레,코스피 7% 급락…역대 최대치 452 낙폭 ‘검은 화요일’,https://www.hani.co.kr/arti/economy/economy_ge...
4,C_001,"'검은 화요일' 코스피, 7% 급락 5,800선 내줘…낙폭 사상 최대(종합)",https://www.yna.co.kr/view/AKR2026030313480000...,25,0.827594,0.577687,2026-03-03 14:09:45+09:00,오피니언뉴스,"코스피, 장 중 5900선 하회...20만전자·100만닉스도 무너져",http://www.opinionnews.co.kr/news/articleView....
...,...,...,...,...,...,...,...,...,...,...
224,C_298,"바닥인 줄 알고 풀베팅했는데…""나 떨고 있니?"" 개미들 비명",https://www.hankyung.com/article/202603048289i,1,0.825861,0.608575,2026-02-03 17:13:35+09:00,한국경제,급락장서 레버리지에 베팅…간 큰 '전투개미' 웃었다,https://www.hankyung.com/article/2026020315671
225,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,2,0.835166,0.574641,2026-03-03 13:11:44+09:00,한국경제,'이란 쇼크'에 코스피 매도 사이드카…'20만전자' 위태,https://www.hankyung.com/article/2026030356316
226,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,3,0.824786,0.569476,2026-03-03 13:52:18+09:00,뉴스퀘스트,중동발 '검은 화요일'…코스피 사이드카 발동·삼성전자 20만원 깨져,https://www.newsquest.co.kr/news/articleView.h...
227,C_327,[ET특징주] 코스피 하락장에 삼성전자·SK하이닉스 '출렁',https://www.etnews.com/20260304000373,5,0.820939,0.699588,2026-03-03 14:22:31+09:00,연합뉴스,[특징주] 이란 사태에 삼성전자 20만원·SK하이닉스 100만원 깨져,https://www.yna.co.kr/view/AKR2026030311050000...


In [12]:
from pathlib import Path
import json
import pandas as pd


SOURCE_NEWS_DIR = Path(
    "./source_data/source_news"
)


def build_source_news_json(
    case_id: str,
):
    """
    LLM User Prompt의 news[]에 들어갈
    최종 뉴스 JSON 파일을 생성합니다.

    입력:
    - representative_articles.parquet
    - news_novelty.parquet

    출력:
    - ./source_data/source_news/{case_id}/source_news.json
    """

    paths = get_case_paths(
        case_id
    )


    representative_path = (
        paths["case_dir"]
        / "representative_articles.parquet"
    )

    novelty_path = (
        paths["case_dir"]
        / "news_novelty.parquet"
    )


    # =========================================
    # 파일 확인
    # =========================================

    if not representative_path.exists():
        raise FileNotFoundError(
            "representative_articles.parquet을 "
            f"찾을 수 없습니다: {representative_path}"
        )


    if not novelty_path.exists():
        raise FileNotFoundError(
            "news_novelty.parquet을 "
            f"찾을 수 없습니다: {novelty_path}"
        )


    # =========================================
    # 데이터 로드
    # =========================================

    representatives_df = pd.read_parquet(
        representative_path
    )

    novelty_df = pd.read_parquet(
        novelty_path
    )


    # =========================================
    # 대표 기사 + novelty 결과 결합
    # =========================================

    final_df = representatives_df.merge(
        novelty_df[
            [
                "cluster_id",
                "representative_article_id",
                "novelty_status",
                "earliest_similar_date",
            ]
        ],

        left_on=[
            "cluster_id",
            "article_id",
        ],

        right_on=[
            "cluster_id",
            "representative_article_id",
        ],

        how="left",

        validate="one_to_one",
    )


    # =========================================
    # 무결성 검사
    # =========================================

    if final_df[
        "novelty_status"
    ].isna().any():

        missing_clusters = (
            final_df.loc[
                final_df[
                    "novelty_status"
                ].isna(),
                "cluster_id",
            ]
            .tolist()
        )

        raise ValueError(
            "novelty 결과가 없는 cluster가 있습니다: "
            f"{missing_clusters}"
        )


    # =========================================
    # 시간 순 정렬
    # =========================================

    final_df = (
        final_df
        .sort_values(
            [
                "published_at_kst",
                "cluster_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # =========================================
    # news[] 생성
    # =========================================

    news_records = []


    for _, row in final_df.iterrows():

        published_at = (
            row["published_at_kst"].isoformat()
            if pd.notna(
                row["published_at_kst"]
            )
            else None
        )


        earliest_similar_date = (
            str(
                row[
                    "earliest_similar_date"
                ]
            )
            if pd.notna(
                row[
                    "earliest_similar_date"
                ]
            )
            else None
        )


        news_records.append(
            {
                "cluster_id":
                    row[
                        "cluster_id"
                    ],

                "source_id":
                    row[
                        "article_id"
                    ],

                "source":
                    row[
                        "publisher_name"
                    ],

                "published_at":
                    published_at,

                "title":
                    row[
                        "title"
                    ],

                "body":
                    row[
                        "normalized_body"
                    ],

                "url":
                    row[
                        "original_link"
                    ],

                "propagation_count":
                    int(
                        row[
                            "propagation_count"
                        ]
                    ),

                "propagation_source_count":
                    int(
                        row[
                            "propagation_source_count"
                        ]
                    ),

                "novelty_status":
                    row[
                        "novelty_status"
                    ],

                "earliest_similar_date":
                    earliest_similar_date,
            }
        )


    # =========================================
    # 최종 JSON 구조
    # =========================================

    source_news = {
        "news":
            news_records
    }


    # =========================================
    # 저장 경로
    # =========================================

    output_dir = (
        SOURCE_NEWS_DIR
        / case_id
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    output_path = (
        output_dir
        / "source_news.json"
    )


    # =========================================
    # JSON 저장
    # =========================================

    with open(
        output_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            source_news,
            f,
            ensure_ascii=False,
            indent=2,
        )


    # =========================================
    # 결과 출력
    # =========================================

    print(
        "case_id:",
        case_id
    )

    print(
        "company:",
        CASES[
            case_id
        ][
            "company"
        ]
    )

    print(
        "최종 news 수:",
        len(
            news_records
        )
    )

    print(
        "저장 경로:",
        output_path
    )


    return source_news

In [13]:
hanmi_source_news = (
    build_source_news_json(
        case_id="hanmi_20260612"
    )
)

case_id: hanmi_20260612
company: 한미반도체
최종 news 수: 8
저장 경로: source_data/source_news/hanmi_20260612/source_news.json


In [14]:
skhynix_source_news = (
    build_source_news_json(
        case_id="skhynix_20260304"
    )
)

case_id: skhynix_20260304
company: SK하이닉스
최종 news 수: 68
저장 경로: source_data/source_news/skhynix_20260304/source_news.json


In [15]:
print(
    json.dumps(
        hanmi_source_news,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "news": [
    {
      "cluster_id": "C_005",
      "source_id": "A_e73325cde85fa9de",
      "source": "한국경제",
      "published_at": "2026-06-11T16:06:14+09:00",
      "title": "'극단적 공포'에 개미들 패닉…증시 뒤흔든 '범인' 따로 있었다",
      "body": "- 기사 스크랩 - 댓글 - 공유 - 글자크기 - 프린트 주가 변동성 키우는 삼전닉스 레버리지 ETF 공포지수 급등은 기본적으로 지정학 갈등과 인공지능(AI) 피크아웃 우려, 채권 금리 상승 등 글로벌 증시를 흔드는 공통 요인 때문이지만 코스피지수의 극심한 변동성은 이것만으로 설명되지 않는 수준이다. 전문가들은 지난달 27일 삼성전자·SK하이닉스 단일종목 레버리지 상장지수펀드(ETF) 상장을 계기로 레버리지 ETF 투자 열풍이 불며 국내 증시의 변동성이 확대됐다는 분석을 내놓고 있다. ‘극단적 공포’ 불러온 레버리지 11일 한국거래소에 따르면 한국형 공포지수로 불리는 VKOSPI는 지난 9일 91.2를 기록했다. 2008년 금융위기 때 기록한 전고점(89.3)을 넘었다. 지난 10일엔 88.4로 소폭 내려왔지만 미국과 이란 간 전쟁 발발 후에 나타난 70~80선을 훌쩍 뛰어넘은 상태다. 이 지수는 50을 넘으면 ‘극단적 공포’ 구간으로 평가된다. 실제 코스피 변동성은 눈에 띄는 수준이다. 지난 8일 코스피가 8%대 급락하면서 매매가 20분간 중단되는 서킷브레이커가 발동한 데 이어, 9일에는 지수가 다시 급반등해 매수 사이드카가 발동했다. 하지만 이튿날 지수는 다시 파란불을 켜면서 매도 사이드카가 나왔다. 한국경제신문이 코스피 일중 변동폭(장중 고가와 저가 차이)을 분석한 결과 올해 1월 월평균 98.7포인트였던 일중 등락폭은 5월 306.6포인트, 6월에는 392.12포인트로 대폭 증가한 것으로 나타났다. 지정학 우려와 반도체 고점 논란 등이 변동성

In [28]:
print(
    json.dumps(
        skhynix_source_news,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "news": [
    {
      "cluster_id": "C_013",
      "source_id": "A_a21c8ae67dfdd122",
      "source": "연합뉴스",
      "published_at": "2026-03-03T15:42:28+09:00",
      "title": "[특징주] 이란사태발 코스피 급락에 '20만전자·100만닉스'도 깨져(종합...",
      "body": "이란 사태 여파로 3일 코스피가 급락하자 '대장주' 삼성전자와 SK하이닉스 주가가 10% 안팎으로 급락했다. SK하이닉스는 키움증권이 110만→130만원, 한국투자증권은 130만→150만원으로 목표주가를 조정했다. 박유악 키움증권 연구원은 올해 삼성전자의 영업이익이 전년 대비 359% 늘어난 200조원, SK하이닉스는 260% 증가한 170조원이 될 것으로 전망했다. 전체 내용을 이해하기 위해서는 기사 본문과 함께 읽어야 합니다. (서울=연합뉴스) 임은진 기자 = 이란 사태 여파로 3일 코스피가 급락하자 '대장주' 삼성전자[005930]와 SK하이닉스[000660] 주가가 10% 안팎으로 급락했다. 한국거래소에 따르면 이날 삼성전자는 전 거래일 대비 9.88% 내린 19만5천100원에 거래를 마쳤다. 삼성전자 주가가 20만원 선 아래로 내린 것은 지난달 24일 이후 처음이다. SK하이닉스는 11.50% 하락한 93만9천원을 기록하면서 주가 100만원 선이 깨졌다. 시가총액은 각각 1천155억원, 669조원으로, 코스피 내 비중은 각각 24.21%, 14.03%가 됐다. 이는 미국과 이스라엘이 이란에 대해 군사 행동을 개시하면서 투자자들이 '위험 자산' 회피에 나선 영향으로 풀이된다. 다만 증권가는 두 반도체 대표 기업의 올해 실적이 개선될 것이라면서 목표주가를 잇달아 상향 조정했다. 삼성전자의 경우 키움증권[039490]이 21만→26만원, iM증권은 20만→26만5천원, 한국투자증권은 22만→27만원으로 각각 올렸다. 그는 양사에 